# TNBC Fusion Clone CNV Additive Analysis: Arm-Aware Segment Level

- Use environment_CNandRNAnotebooks.yml
- Edit the 1st cell below to run for either HCC1806 or MDA-MB-231

This notebook tests whether fusion clone copy number (CN) profiles are consistent with a
simple additive combination of their two parental control clones:

    expected additive CN = CN(parent A) + CN(parent B)

Control-FREEC's CopyNumber calls are already integer-rounded, so segment-level residuals
are always exact integers. For each fusion clone, every genomic segment is classified as:

- **match**: fusion CN is exactly equal to the additive expectation
- **over**: fusion CN is higher than the additive expectation
- **under**: fusion CN is lower than the additive expectation

Two separate confidence flags, low bin count and proximity to a rounding boundary in the
underlying continuous signal, are reported alongside every segment without changing its
classification, since those are questions about how much to trust a given integer call,
not about how much deviation to forgive.

**Analysis structure:**

1. Merge FREEC copy number data across all samples
2. Quality filtering: remove unmappable bins, apply the ENCODE blacklist, merge MedianRatio,
   and correct for bins Control-FREEC left as CopyNumber = 0 rather than -1 despite Ratio
   and MedianRatio both being the -1 unmappable sentinel
3. Genomic coordinate helpers: centromere positions, chromosome arm labels (sorted in true
   genomic order, not alphabetically), per-sample complexity diagnostic across all 9
   individual samples
4. Build arm-aware consensus segments (segments never span a centromere)
5. Additive expectation and residuals
6. Segment classification (exact-integer match / over / under; informative segments are
   flagged but not attributed to a specific parent)
7. Segment confidence flags (low bin count, proximity to a rounding boundary, overlap with
   a parental subclone region from a low-threshold Control-FREEC re-run)
8. Continuous (non-rounded), log2-scale copy-number residual, for later correlation against
   external log2 fold-change data (e.g. RNA-seq)
9. Genome-wide quantification (base-pair weighted fractions, weighted MAE, weighted mean
   signed residual)
10. Bootstrap confidence intervals (chromosome-block resampling)
11. Test for a systematic genome-wide gain or loss bias
12. Chromosome-arm-level rollup
13. Confidence flag coverage by arm (how much of each arm's call rests on flagged segments)
14. Plotting functions, with an automatic, data-driven switch to a symlog axis on the
    scatter and residual plots when an extreme value is actually present
15. Extreme segment diagnostic (checking unusually high or low segments for artifacts)
16. Master runner that ties all of the above together for one fusion clone
17. Run the analysis for each of the three fusion clones
18. Cross-clone comparison and recurrence

Note: this notebook intentionally does not repeat general exploratory QC plots
(whole-cohort heatmaps, etc.) from earlier notebooks, with one exception --
Part 2.2 includes a hierarchical clustering check, since whether each
fusion clone groups with its own two parents is a direct test of the additive
model itself, not just general QC. It is otherwise focused specifically on the
additive-inheritance question.


---
# Part 0 - Imports and Configuration

In [ ]:
#(Make sure to comment out the set of lines below for the cell line not being analyzed)

## ---------- Use the lines below for HCC1806 --------------
FREEC_BASE_DIR = "/stor/work/Brock/kennedy/SC_repo/data/MatchedTrioAdditiveCNVAnalysis/MatchedTrioFREEC_HCC1806"
FUSION_PAIRS = {
    "C2C4_1806": ("C2_1806_PT", "C4_1806_PT"),
    "C5C2_1806": ("C5_1806_PT", "C2_1806_PT"),
    "C6C7_1806": ("C6_1806_PT", "C7_1806_PT"),
}
SAMPLE_LABELS = {
    "C2C4_1806": "Fusion\nC2C4",
    "C2_1806_PT": "Control\nC2",
    "C4_1806_PT": "Control\nC4",
    "C5C2_1806": "Fusion\nC5C2",
    "C5_1806_PT": "Control\nC5",
    "C6C7_1806": "Fusion\nC6C7",
    "C6_1806_PT": "Control\nC6",
    "C7_1806_PT": "Control\nC7"
}

## ---------- Use the lines below for MDA-MB-231 --------------
# FREEC_BASE_DIR = "/stor/work/Brock/kennedy/SC_repo/data/MatchedTrioAdditiveCNVAnalysis/MatchedTrioFREEC_MDA-MB-231"
# FUSION_PAIRS = {
#     "C1C4_231": ("C1_231_PT", "C4_231_PT"),
#     "C5C3_231": ("C5_231_PT", "C3_231_PT"),
#     "C6C8_231": ("C6_231_PT", "C8_231_PT"),
# }
# SAMPLE_LABELS = {
#     "C1C4_231": "Fusion\nC1C4",
#     "C1_231_PT": "Control\nC1", 
#     "C4_231_PT": "Control\nC4",
#     "C5C3_231": "Fusion\nC5C3",
#     "C5_231_PT": "Control\nC5",
#     "C3_231_PT": "Control\nC3",
#     "C6C8_231": "Fusion\nC6C8",
#     "C6_231_PT": "Control\nC6",
#     "C8_231_PT": "Control\nC8"
# }

In [ ]:
import os
import glob
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns  # only used for the Part 2.2 clustermap
from scipy.stats import wilcoxon
from IPython.display import display

pd.set_option("display.max_columns", 50)

# Chromosomes to keep (autosomes 1-22 plus X)
ALLOWED_CHROMS = [str(i) for i in range(1, 23)] + ["X"]

# FREEC window size in bp, must match the window used in the FREEC config
WINDOW_SIZE = 500000

# Reference genome build used for alignment and FREEC (confirmed hg38)
GENOME = "hg38"

# Output directory for merged data and per-clone results
BASE_OUTPUT_DIR = "arm_level_inheritance_outputs"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

# ENCODE blacklist for hg38, used to remove low-mappability and repeat-rich regions
BLACKLIST_URL = "https://github.com/Boyle-Lab/Blacklist/raw/master/lists/hg38-blacklist.v2.bed.gz"
BLACKLIST_LOCAL = os.path.join(BASE_OUTPUT_DIR, "hg38-blacklist.v2.bed.gz")

# All parental control samples appearing in FUSION_PAIRS (used only for
# the subclone-region overlap flag in Section 7).
PARENTAL_SAMPLES = sorted({sample for pair in FUSION_PAIRS.values() for sample in pair})

# Colors used consistently across every plot in this notebook
CLASS_COLORS = {"match": "0.65", "over": "firebrick", "under": "steelblue"}

print("Configuration loaded.")
print("Fusion pairs:", list(FUSION_PAIRS.keys()))


### Figure Style Settings

Global font and figure-saving configuration used by every plot in this
notebook. `save_fig(fig, name, out_dir=None)` is the single figure-saving
function used everywhere below (defaulting to `BASE_OUTPUT_DIR`, or pass
a specific subdirectory such as `cross_clone_dir` or a per-clone
`output_dir`). Change `FIG_EXT` to `".svg"` to save vector figures instead.

In [ ]:
from matplotlib import rcParams

# FONT_FAMILY = "Arial"
FONT_SIZE   = 16
FONT_BOLD   = False
FIG_EXT     = ".png"   # ".svg" or ".png"
FIG_DPI     = 300       # only used when FIG_EXT == ".png"

def apply_style():
    w = "bold" if FONT_BOLD else "normal"
    rcParams.update({
        # "font.family":           FONT_FAMILY,
        "font.size":             FONT_SIZE,
        "font.weight":           w,
        "axes.titlesize":        FONT_SIZE + 1,
        "axes.titleweight":      w,
        "axes.labelsize":        FONT_SIZE,
        "axes.labelweight":      w,
        "xtick.labelsize":       FONT_SIZE - 1,
        "ytick.labelsize":       FONT_SIZE - 1,
        "legend.fontsize":       FONT_SIZE - 1,
        "legend.title_fontsize": FONT_SIZE,
        "figure.titlesize":      FONT_SIZE + 2,
        "figure.titleweight":    w,
    })

apply_style()

def save_fig(fig, name, out_dir=None, tight=True):
    """
    Save a figure using the extension/DPI configured above. out_dir defaults
    to BASE_OUTPUT_DIR; pass a specific subdirectory (e.g. cross_clone_dir,
    or a per-clone output_dir) to save there instead.
    """
    out_dir = out_dir if out_dir is not None else BASE_OUTPUT_DIR
    path = os.path.join(out_dir, f"{name}{FIG_EXT}")
    kw = {"bbox_inches": "tight"} if tight else {}
    if FIG_EXT.lower() == ".png":
        kw["dpi"] = FIG_DPI
    fig.savefig(path, **kw)
    print(f"Saved: {path}")
    return path

print(f"Style applied  |  FIG_EXT={FIG_EXT}  |  FONT_SIZE={FONT_SIZE}  |  FONT_BOLD={FONT_BOLD}")


---
# Part 0b - Group Definitions and Alternate Sample Labels

**Alternate sample labels:** `SAMPLE_LABELS` lets you give any sample a custom
display label used in every plot title, axis, and legend in this notebook,
without changing the underlying column name used for data lookups (fusion/parent
matching, dataframe columns, file paths all keep using the real sample name).
Example: `"C2C4_1806": "Fusion C2C4"`. Any sample not given a custom label falls
back to a short default (the text before the first underscore).

**Group definitions:** `SAMPLE_GROUP` / `get_group()` classify each sample as
Control ("C") or Fusion ("F"). `GROUP_COLORS` and `SAMPLE_GROUP_ORDER` set the
color and sort order used for these groups in Part 2 (the cross-clone
hierarchical clustering and per-chromosome heatmap).


In [ ]:
def get_label(sample_name):
    """Return the display label for a sample: SAMPLE_LABELS override if
    present, else the default short name (text before the first underscore)."""
    if sample_name in SAMPLE_LABELS:
        return SAMPLE_LABELS[sample_name]
    return sample_name.split("_")[0]

# Explicit sample -> group lookup, built from FUSION_PAIRS. This is looked up
# by exact name rather than parsed from a prefix, since a fusion clone name
# like "C2C4_1806" would otherwise be indistinguishable from a control clone
# name by prefix alone.
SAMPLE_GROUP = {}
for _s in PARENTAL_SAMPLES:
    SAMPLE_GROUP[_s] = "C"
for _f in FUSION_PAIRS:
    SAMPLE_GROUP[_f] = "F"

def get_group(sample_name):
    """Return 'C' (control clone) or 'F' (fusion clone) for a sample name,
    via the explicit SAMPLE_GROUP lookup above."""
    return SAMPLE_GROUP.get(sample_name, "?")

GROUP_COLORS = {"C": "black", "F": "darkorange"}
SAMPLE_GROUP_ORDER = ["C", "F"]

print("Extended configuration loaded.")
print(f"Control clones (n={len(PARENTAL_SAMPLES)}): {PARENTAL_SAMPLES}")
print(f"Fusion clones (n={len(FUSION_PAIRS)}): {list(FUSION_PAIRS.keys())}")


---
# Data Import and Processing

Loads and merges the raw FREEC output, applies quality filtering, and builds
the genomic coordinate helpers everything below depends on. Unlike the
Functions section that follows, these cells run once and build shared state
(`cn_df`, `blacklist_df`, `centromeres`, etc.) rather than just defining
functions to be called later.


## 1. Merge FREEC Copy Number Data

Reads every `_ratio.txt` file produced by Control-FREEC and pulls out the
`Chromosome`, `Start`, and `CopyNumber` columns for each sample, then merges
all samples into one wide bin-level matrix.

In [ ]:
def merge_freec_cn_data(base_dir, allowed_chroms, output_tsv):
    """
    Find every _ratio.txt file produced by Control-FREEC under base_dir,
    pull out the Chromosome, Start, and CopyNumber columns, and merge all
    samples into one wide matrix (one column per sample).
    """
    ratio_files = glob.glob(os.path.join(base_dir, "all_ratio_files*", "*_ratio.txt"))
    if len(ratio_files) == 0:
        raise FileNotFoundError(f"No _ratio.txt files found under {base_dir}")

    dfs = []
    for file in ratio_files:
        base = os.path.basename(file)
        sample_name = base.split("_paired")[0]

        df = pd.read_csv(file, sep="\t", usecols=["Chromosome", "Start", "CopyNumber"],
                         dtype={"Chromosome": str})
        df = df[df["Chromosome"].isin(allowed_chroms)]
        df = df.rename(columns={"CopyNumber": sample_name})
        dfs.append(df)

    merged_df = dfs[0]
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=["Chromosome", "Start"], how="inner")

    chrom_order = {chrom: i for i, chrom in enumerate(allowed_chroms)}
    merged_df["Chromosome_sort"] = merged_df["Chromosome"].map(chrom_order)
    merged_df = merged_df.sort_values(["Chromosome_sort", "Start"]).drop(columns="Chromosome_sort")
    merged_df = merged_df.reset_index(drop=True)

    merged_df.to_csv(output_tsv, sep="\t", index=False)
    sample_cols_found = [c for c in merged_df.columns if c not in ["Chromosome", "Start"]]
    print(f"Merged CN matrix saved to {output_tsv}")
    print(f"Samples found: {sample_cols_found}")
    return merged_df


In [ ]:
merged_cn_path = os.path.join(BASE_OUTPUT_DIR, "merged_CN_matrix.tsv")
merged_cn_df = merge_freec_cn_data(FREEC_BASE_DIR, ALLOWED_CHROMS, merged_cn_path)
display(merged_cn_df.head())


## 2. Quality Filtering

### 2.1 Remove Unmappable Bins

Control-FREEC marks bins it could not confidently call with a -1 sentinel value.
Any bin where any sample has a -1 is dropped from all samples.

In [ ]:
def remove_unmappable_bins(df):
    """
    Drop any bin where any sample has a -1 sentinel value (FREEC unmappable flag).
    """
    sample_cols = [c for c in df.columns if c not in ["Chromosome", "Start"]]
    mask_bad = (df[sample_cols] == -1).any(axis=1)
    cleaned = df[~mask_bad].copy().reset_index(drop=True)
    print(f"Bins before -1 filter: {len(df)}")
    print(f"Bins after -1 filter: {len(cleaned)} (removed {mask_bad.sum()})")
    return cleaned

cn_df = remove_unmappable_bins(merged_cn_df)


### 2.2 ENCODE Blacklist Filtering

Removes pericentromeric, telomeric, and other low-mappability or repeat-rich
regions that can look like copy number changes but are really mapping
artifacts. This is standard practice for low-pass WGS copy number data
(see Amemiya et al. 2019 for the blacklist itself, and Scheinin et al. 2014 /
the QDNAseq convention for its use in CN analysis). FREEC's own -1 filter does
not catch everything this blacklist catches.

In [ ]:
def load_blacklist(url=BLACKLIST_URL, local_path=BLACKLIST_LOCAL):
    """
    Load the ENCODE blacklist BED file. Downloads once and caches locally.
    """
    if not os.path.exists(local_path):
        print(f"Downloading blacklist from {url}")
        urllib.request.urlretrieve(url, local_path)
        print(f"Saved to {local_path}")
    bl = pd.read_csv(local_path, sep="\t", header=None, compression="gzip",
                     usecols=[0, 1, 2], names=["chrom", "start", "end"])
    bl["chrom_clean"] = bl["chrom"].str.replace("chr", "", regex=False)
    print(f"Blacklist loaded: {len(bl)} regions")
    return bl

def filter_blacklist(df, blacklist_df, window_size=WINDOW_SIZE):
    """
    Remove bins from df that overlap any blacklisted region.
    df must have Chromosome and Start columns (Start is 0-based).
    """
    result_parts = []
    for chrom, bin_grp in df.groupby("Chromosome"):
        c = str(chrom).replace("chr", "")
        bl = blacklist_df[blacklist_df["chrom_clean"] == c]
        if bl.empty:
            result_parts.append(bin_grp)
            continue
        starts = bin_grp["Start"].values.astype(int)
        ends = starts + window_size
        keep = np.ones(len(starts), dtype=bool)
        bl_starts = bl["start"].values.astype(int)
        bl_ends = bl["end"].values.astype(int)
        for bs, be in zip(bl_starts, bl_ends):
            keep &= ~((bs < ends) & (be > starts))
        result_parts.append(bin_grp[keep])
    filtered = pd.concat(result_parts).reset_index(drop=True)
    return filtered

blacklist_df = load_blacklist()
n_before = len(cn_df)
cn_df = filter_blacklist(cn_df, blacklist_df, window_size=WINDOW_SIZE)
print(f"Bins after blacklist filter: {len(cn_df)} (removed {n_before - len(cn_df)})")


### 2.3 Merge MedianRatio Data

MedianRatio is Control-FREEC's own smoothed, segmented ratio value, the
continuous signal that CopyNumber was rounded from. This is loaded
separately from CopyNumber, restricted to the bins already retained in
cn_df so far, and used both for the correction in the next step and later
for the rounding-boundary confidence check (section 7 below). Any -1
sentinel values in MedianRatio itself are dropped as invalid before the
multi-sample merge, so a bin only survives here if every sample has a
usable MedianRatio.

In [ ]:
def merge_freec_median_ratio_data(base_dir, allowed_chroms, retained_bins_df):
    """
    Find every _ratio.txt file produced by Control-FREEC under base_dir,
    pull out the Chromosome, Start, and MedianRatio columns, merge all
    samples into one wide matrix, and restrict to the bins already
    retained in retained_bins_df (the filtered CN matrix so far).
    """
    ratio_files = glob.glob(os.path.join(base_dir, "all_ratio_files*", "*_ratio.txt"))
    if len(ratio_files) == 0:
        raise FileNotFoundError(f"No _ratio.txt files found under {base_dir}")

    dfs = []
    for file in ratio_files:
        base = os.path.basename(file)
        sample_name = base.split("_paired")[0]

        df = pd.read_csv(file, sep="\t", usecols=["Chromosome", "Start", "MedianRatio"],
                         dtype={"Chromosome": str})
        df = df[df["Chromosome"].isin(allowed_chroms)]
        # A MedianRatio of -1 is FREEC's own sentinel for a bin with no
        # usable signal. Such rows should not feed the confidence check.
        df = df[df["MedianRatio"] >= 0]
        df = df.rename(columns={"MedianRatio": sample_name})
        dfs.append(df)

    merged_df = dfs[0]
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=["Chromosome", "Start"], how="inner")

    keep_bins = retained_bins_df[["Chromosome", "Start"]]
    merged_df = pd.merge(merged_df, keep_bins, on=["Chromosome", "Start"], how="inner")
    merged_df = merged_df.reset_index(drop=True)

    print(f"MedianRatio matrix merged and restricted to retained bins: {len(merged_df)} bins")
    return merged_df


In [ ]:
median_ratio_df = merge_freec_median_ratio_data(FREEC_BASE_DIR, ALLOWED_CHROMS, cn_df)


### 2.4 Correct for MedianRatio Sentinel Bins Not Caught by the CopyNumber Filter

Control-FREEC does not always use -1 consistently across its columns for an
unmappable bin. A bin can have Ratio and MedianRatio both equal to -1 (no
usable signal) while CopyNumber is reported as 0 rather than -1, which would
make it look like a real homozygous deletion call to the section 2.1 filter
rather than the no-data bin it actually is. Since median_ratio_df already
excludes any bin where any sample's MedianRatio was a -1 sentinel, cn_df is
now restricted to match that same bin set, catching this case directly.

In [ ]:
n_before = len(cn_df)
good_bins = median_ratio_df[["Chromosome", "Start"]]
cn_df = pd.merge(cn_df, good_bins, on=["Chromosome", "Start"], how="inner").reset_index(drop=True)
print(f"Bins after MedianRatio sentinel correction: {len(cn_df)} (removed {n_before - len(cn_df)})")

filtered_cn_path = os.path.join(BASE_OUTPUT_DIR, "filtered_CN_matrix.tsv")
cn_df.to_csv(filtered_cn_path, sep="\t", index=False)
print(f"Saved filtered CN matrix to {filtered_cn_path}")

median_ratio_path = os.path.join(BASE_OUTPUT_DIR, "median_ratio_matrix.tsv")
median_ratio_df.to_csv(median_ratio_path, sep="\t", index=False)
print(f"Saved MedianRatio matrix to {median_ratio_path}")


### 2.5 Quick QC Summary

In [ ]:
def quick_qc_summary(df, sample_cols):
    rows = []
    for col in sample_cols:
        s = df[col]
        rows.append({
            "Sample": col,
            "n_bins": s.shape[0],
            "min_CN": s.min(),
            "max_CN": s.max(),
            "mean_CN": s.mean(),
            "median_CN": s.median(),
            "std_CN": s.std(),
        })
    return pd.DataFrame(rows)

all_sample_cols = [c for c in cn_df.columns if c not in ["Chromosome", "Start"]]
qc_summary = quick_qc_summary(cn_df, all_sample_cols)
display(qc_summary)
qc_summary.to_csv(os.path.join(BASE_OUTPUT_DIR, "qc_summary.tsv"), sep="\t", index=False)


## 3. Genomic Coordinate Helpers

### 3.1 Centromere Positions

Centromere midpoint per chromosome, used to define chromosome arm boundaries.
Fetched from UCSC for the confirmed genome build (hg38), with a hardcoded
fallback in case the download is unavailable.

In [ ]:
CENTROMERE_URLS = {
    "hg38": "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/centromeres.txt.gz",
    "hg19": "https://hgdownload.soe.ucsc.edu/goldenPath/hg19/database/gap.txt.gz",
}

HG38_CENTROMERE_FALLBACK = {
    "1": 121500000, "2": 92000000, "3": 90900000, "4": 49700000, "5": 46500000,
    "6": 58800000, "7": 60100000, "8": 45200000, "9": 43000000, "10": 39800000,
    "11": 53400000, "12": 35500000, "13": 17700000, "14": 17200000, "15": 19000000,
    "16": 36800000, "17": 25100000, "18": 18500000, "19": 26200000, "20": 28100000,
    "21": 12000000, "22": 15000000, "X": 61000000,
}

def load_centromeres(genome=GENOME):
    """
    Load centromere midpoint position per chromosome from UCSC.
    Falls back to a hardcoded hg38 table if the download fails.
    """
    try:
        url = CENTROMERE_URLS[genome]
        raw = pd.read_csv(url, sep="\t", header=None, compression="gzip")
        if genome == "hg38":
            raw.columns = ["bin", "chrom", "start", "end", "name"]
        else:
            raw.columns = ["bin", "chrom", "start", "end", "ix", "n", "size", "type", "bridge"]
            raw = raw[raw["type"] == "centromere"]
        raw["chrom_clean"] = raw["chrom"].str.replace("chr", "", regex=False)
        centro = {}
        for chrom, grp in raw.groupby("chrom_clean"):
            centro[chrom] = int((grp["start"].min() + grp["end"].max()) / 2)
        print(f"Centromere positions loaded from UCSC ({genome}): {len(centro)} chromosomes")
        return centro
    except Exception as err:
        print(f"UCSC centromere fetch failed ({err}). Using hardcoded hg38 fallback.")
        return dict(HG38_CENTROMERE_FALLBACK)

centromeres = load_centromeres(GENOME)


### 3.2 Chromosome Arm Labeling

Labels a genomic region as being on the p arm, q arm, or as spanning the
centromere. This is a general-purpose helper for labeling arbitrary regions.
The arm-aware segment builder in section 4 below does not call this function
directly. It instead tracks which side of the centromere each bin falls on
as segments are built, and labels the finished segment from that, since a
single 500 kb bin can itself straddle the centromere position, which would
otherwise make a start/end based label disagree with where the segment was
actually split.

In [ ]:
def get_arm(chrom, start, end, centromeres):
    """
    Label a region as p arm, q arm, or spanning the centromere.
    """
    c = str(chrom).replace("chr", "")
    if c not in centromeres:
        return f"{c}?"
    cen = centromeres[c]
    if int(end) < cen:
        return f"{c}p"
    if int(start) > cen:
        return f"{c}q"
    return f"{c}centromere"


### 3.3 Genome-Wide Coordinate Helpers (for whole-genome plots)

In [ ]:
def add_genome_x(df, allowed_chroms):
    """
    Add a genome_x column giving each bin a running genome-wide coordinate,
    and return chromosome center positions for axis tick labels.
    """
    out = df.copy()
    chrom_max = out.groupby("Chromosome", observed=True)["Start"].max()
    offsets = {}
    running = 0
    for chrom in allowed_chroms:
        if chrom in chrom_max.index:
            offsets[chrom] = running
            running += chrom_max.loc[chrom]
    out["genome_x"] = out.apply(lambda r: r["Start"] + offsets[r["Chromosome"]], axis=1)
    chrom_centers = {chrom: offsets[chrom] + chrom_max.loc[chrom] / 2 for chrom in chrom_max.index}
    return out, chrom_centers

def add_segment_genome_coords(seg_df, allowed_chroms):
    """
    Add genome_start and genome_end columns for a segment-level dataframe,
    plus chromosome center positions and offsets for axis labels.
    """
    out = seg_df.copy()
    chrom_max = out.groupby("Chromosome", sort=False)["End"].max()
    offsets = {}
    running = 0
    for chrom in allowed_chroms:
        if chrom in chrom_max.index:
            offsets[chrom] = running
            running += chrom_max.loc[chrom]
    out["genome_start"] = out["Start"] + out["Chromosome"].map(offsets)
    out["genome_end"] = out["End"] + out["Chromosome"].map(offsets)
    chrom_centers = {chrom: offsets[chrom] + chrom_max.loc[chrom] / 2 for chrom in chrom_max.index}
    return out, chrom_centers, offsets

def build_arm_order(allowed_chroms):
    """
    Build the canonical genomic ordering of chromosome arms (1p, 1q, 2p,
    2q, ..., 22p, 22q, Xp, Xq), used instead of Python's default string
    sort, which would place '10p' before '2p' since it compares arm labels
    character by character rather than numerically by chromosome.
    """
    order = []
    for chrom in allowed_chroms:
        order.append(f"{chrom}p")
        order.append(f"{chrom}q")
    return order

ARM_ORDER = build_arm_order(ALLOWED_CHROMS)

def sort_arms(arms):
    """
    Sort a collection of chromosome arm labels into genomic order. Any
    label not in the canonical p/q ordering (for example an unresolved
    '?' or 'centromere' label) is appended at the end, alphabetically.
    """
    arm_rank = {arm: i for i, arm in enumerate(ARM_ORDER)}
    known = sorted([a for a in arms if a in arm_rank], key=lambda a: arm_rank[a])
    unknown = sorted([a for a in arms if a not in arm_rank])
    return known + unknown


### 3.4 Per-Sample Complexity Diagnostic

Segment count and percent genome altered for each of the 8 individual
samples on its own, not the 3 fusion trios. This checks whether any one
parental clone (or fusion clone) stands out as unusually rearranged before
that gets attributed to the fusion itself. A sample with a high segment
count relative to the others is more fragmented, and this could be real
biology, or could reflect noisier data for that particular library. Percent
genome altered is computed relative to that sample's own most common
(highest base-pair-weighted) copy number state, not a fixed assumed
baseline, since parental and fusion samples are expected to sit at
different baseline ploidies.

In [ ]:
def build_single_sample_segments(bin_df, sample, centromeres, allowed_chroms, round_digits=0):
    """
    Same logic as build_arm_aware_segments, but for one sample only. Used
    for the per-sample complexity diagnostic, where each of the 8 samples
    needs its own segmentation rather than the 3-sample consensus used for
    the inheritance analysis itself.
    """
    df = bin_df[["Chromosome", "Start", sample]].copy()

    chrom_order = {chrom: i for i, chrom in enumerate(allowed_chroms)}
    df["Chromosome_sort"] = df["Chromosome"].map(chrom_order)
    df = df.sort_values(["Chromosome_sort", "Start"]).drop(columns="Chromosome_sort").reset_index(drop=True)

    bin_size_map = {}
    for chrom, g in df.groupby("Chromosome", sort=False):
        starts = g["Start"].sort_values().values
        diffs = np.diff(starts)
        diffs = diffs[diffs > 0]
        bin_size_map[chrom] = int(np.median(diffs)) if len(diffs) > 0 else WINDOW_SIZE

    def arm_side(row):
        cen = centromeres.get(row["Chromosome"])
        if cen is None:
            return "na"
        return "p" if row["Start"] < cen else "q"

    df["arm_side"] = df.apply(arm_side, axis=1)
    df["state"] = df[sample].round(round_digits)

    change_list = []
    for chrom, g in df.groupby("Chromosome", sort=False):
        state_changed = g["state"].ne(g["state"].shift())
        arm_changed = g["arm_side"].ne(g["arm_side"].shift())
        changed = (state_changed | arm_changed).astype(int)
        changed.iloc[0] = 1
        change_list.append(changed)

    df["segment_change"] = pd.concat(change_list).sort_index()
    df["Segment_ID"] = df.groupby("Chromosome", sort=False)["segment_change"].cumsum()

    segment_df = df.groupby(["Chromosome", "Segment_ID"], sort=False).agg(
        Start=("Start", "min"),
        Last_bin_start=("Start", "max"),
        n_bins=("Start", "size"),
        arm_side=("arm_side", "first"),
        **{sample: (sample, "mean")}
    ).reset_index()

    segment_df["BinSize"] = segment_df["Chromosome"].map(bin_size_map)
    segment_df["End"] = segment_df["Last_bin_start"] + segment_df["BinSize"] - 1
    segment_df["segment_bp"] = segment_df["n_bins"] * segment_df["BinSize"]

    return segment_df

def compute_sample_complexity_diagnostics(bin_df, samples, centromeres, allowed_chroms):
    """
    For each sample, segment its own copy number profile independently and
    report the number of segments and the percent of the genome that
    differs from that sample's own dominant copy number state.
    """
    rows = []
    for sample in samples:
        seg = build_single_sample_segments(bin_df, sample, centromeres, allowed_chroms)
        total_bp = seg["segment_bp"].sum()
        bp_by_cn = seg.groupby(sample)["segment_bp"].sum()
        baseline_cn = bp_by_cn.idxmax()
        altered_bp = total_bp - bp_by_cn.loc[baseline_cn]
        rows.append({
            "Sample": sample,
            "n_segments": len(seg),
            "baseline_CN": baseline_cn,
            "percent_genome_altered": 100 * altered_bp / total_bp if total_bp else np.nan,
        })
    return pd.DataFrame(rows)


In [ ]:
all_individual_samples = sorted(set(
    [pa for pa, pb in FUSION_PAIRS.values()] +
    [pb for pa, pb in FUSION_PAIRS.values()] +
    list(FUSION_PAIRS.keys())
))

complexity_diagnostic = compute_sample_complexity_diagnostics(cn_df, all_individual_samples, centromeres, ALLOWED_CHROMS)
complexity_diagnostic = complexity_diagnostic.sort_values("n_segments", ascending=False).reset_index(drop=True)
display(complexity_diagnostic)
complexity_diagnostic.to_csv(os.path.join(BASE_OUTPUT_DIR, "per_sample_complexity_diagnostic.tsv"), sep="\t", index=False)


---
# Functions

Defines every function used to build, classify, and plot one fusion clone's
additive-inheritance analysis. Nothing in this section executes on its own --
it's all called from Part 1 (Run the Analysis for Each Fusion Clone) onward,
via the master runner (16).


## 4. Arm-Aware Segment Building

Collapses per-bin copy number calls into consensus segments. A new segment
starts whenever the rounded copy number of parent A, parent B, or the fusion
changes, or when the bin crosses the centromere of its chromosome, so that no
segment spans two chromosome arms even if the copy number happens to agree
across the centromere.

In [ ]:
def build_arm_aware_segments(bin_df, parent_a, parent_b, fusion, centromeres,
                             allowed_chroms, round_digits=0, output_tsv=None):
    """
    Collapse per-bin copy number calls into consensus segments.
    A new segment starts whenever the rounded copy number of any of
    parent_a, parent_b, or fusion changes, or when the bin crosses the
    centromere of its chromosome, so no segment spans two chromosome arms.
    """
    samples = [parent_a, parent_b, fusion]
    df = bin_df[["Chromosome", "Start"] + samples].copy()

    chrom_order = {chrom: i for i, chrom in enumerate(allowed_chroms)}
    df["Chromosome_sort"] = df["Chromosome"].map(chrom_order)
    df = df.sort_values(["Chromosome_sort", "Start"]).drop(columns="Chromosome_sort").reset_index(drop=True)

    bin_size_map = {}
    for chrom, g in df.groupby("Chromosome", sort=False):
        starts = g["Start"].sort_values().values
        diffs = np.diff(starts)
        diffs = diffs[diffs > 0]
        bin_size_map[chrom] = int(np.median(diffs)) if len(diffs) > 0 else WINDOW_SIZE

    def arm_side(row):
        cen = centromeres.get(row["Chromosome"])
        if cen is None:
            return "na"
        return "p" if row["Start"] < cen else "q"

    df["arm_side"] = df.apply(arm_side, axis=1)

    for sample in samples:
        state_col = f"{sample}__state"
        df[state_col] = df[sample].round(round_digits)

    state_cols = [f"{sample}__state" for sample in samples]

    change_list = []
    for chrom, g in df.groupby("Chromosome", sort=False):
        state_changed = g[state_cols].ne(g[state_cols].shift()).any(axis=1)
        arm_changed = g["arm_side"].ne(g["arm_side"].shift())
        changed = (state_changed | arm_changed).astype(int)
        changed.iloc[0] = 1
        change_list.append(changed)

    df["segment_change"] = pd.concat(change_list).sort_index()
    df["Segment_ID"] = df.groupby("Chromosome", sort=False)["segment_change"].cumsum()

    segment_df = df.groupby(["Chromosome", "Segment_ID"], sort=False).agg(
        Start=("Start", "min"),
        Last_bin_start=("Start", "max"),
        n_bins=("Start", "size"),
        arm_side=("arm_side", "first"),
        **{sample: (sample, "mean") for sample in samples}
    ).reset_index()

    segment_df["BinSize"] = segment_df["Chromosome"].map(bin_size_map)
    segment_df["End"] = segment_df["Last_bin_start"] + segment_df["BinSize"] - 1
    segment_df["Midpoint"] = (segment_df["Start"] + segment_df["End"]) / 2
    segment_df["segment_bp"] = segment_df["n_bins"] * segment_df["BinSize"]

    # Label each segment from the same arm_side value used to force the
    # centromere break above, rather than re-deriving p or q from the
    # segment Start/End against the centromere position. Re-deriving it
    # with get_arm() can disagree at the boundary bin, since a single
    # 500 kb bin can straddle the centromere position itself.
    def arm_label(row):
        if row["arm_side"] == "na":
            return f"{row['Chromosome']}?"
        return f"{row['Chromosome']}{row['arm_side']}"

    segment_df["Arm"] = segment_df.apply(arm_label, axis=1)
    segment_df = segment_df.drop(columns=["arm_side"])

    segment_df = segment_df[
        ["Chromosome", "Arm", "Segment_ID", "Start", "End", "Midpoint", "n_bins", "segment_bp"] + samples
    ].copy()

    if output_tsv is not None:
        segment_df.to_csv(output_tsv, sep="\t", index=False)

    return segment_df


## 5. Expected Additive Model

Adds the additive expectation (parent A plus parent B) and the residual
(fusion minus expected additive). (An earlier version of this notebook also
kept two single-parent-doubling values here for a parent-of-origin
attribution step; that step has been removed -- see the Section 6 note below
-- so those columns are no longer computed.) The single-parent-doubling values
(2 times parent A, 2 times parent B) are also kept, but only used later for
parent-of-origin attribution on informative segments, not as competing
genome-wide models.

In [ ]:
def add_expected_models(seg_df, parent_a, parent_b, fusion):
    """
    Add the additive expectation and the residual of the fusion from the
    additive expectation.
    """
    out = seg_df.copy()
    out["expected_additive"] = out[parent_a] + out[parent_b]
    out["residual"] = out[fusion] - out["expected_additive"]
    return out


## 6. Segment Classification

Control-FREEC's CopyNumber column is already integer-rounded. Every bin
inside one of our consensus segments therefore has an identical integer
value, and the segment-level value (the mean across its bins) collapses to
that same exact integer. This means expected_additive and residual are
always exact integers too, never a fraction. The smallest possible nonzero
residual is exactly 1 copy, and there is no meaningful in-between value for
a tolerance to occupy. Classification is therefore exact-integer equality,
not a calibrated tolerance: match means residual is exactly 0, over means
residual is a positive integer, under means residual is a negative integer.
The same logic applies to informative segments, defined simply as parent A
and parent B not being exactly equal. Note that all segments, not just
informative ones, are used for the match / over / under classification,
since a balanced region (parents equal) can still deviate from its additive
expectation and that deviation is a real observation.

`is_informative` is kept as a simple factual flag (the parents differ here)
and nothing more. An earlier version of this notebook additionally
attributed non-matching informative segments to "parent A like" or
"parent B like" by comparing the fusion's value to 2 x each parent's copy
number. That comparison assumed a mechanism (something closer to
endoreduplication or mitotic slippage within one parental genome) that
does not correspond to how these fusion clones actually arose (fusion of
two independent, already-aneuploid cells), had no fit-quality threshold
(a segment could be labeled "like" a parent it barely resembled, simply
because it was the closer of two equally poor options), and was not
even internally consistent across the three fusion pairs. It has been
removed rather than patched.

In [ ]:
def classify_segments(seg_df, parent_a, parent_b, fusion):
    """
    Classify every segment as match, over, or under using exact integer
    equality, and flag informative segments (parents not exactly equal).
    is_informative is a plain factual flag only -- it is not used here to
    attribute a segment's deviation to a specific parent (see the Section 6
    markdown note for why that step was removed).
    """
    out = seg_df.copy()

    def classify(resid):
        if resid == 0:
            return "match"
        return "over" if resid > 0 else "under"

    out["classification"] = out["residual"].apply(classify)
    out["is_informative"] = out[parent_a] != out[parent_b]

    return out


## 7. Segment Confidence Flags

Since classification is now exact-integer equality rather than a tolerance
band, the question of measurement noise moves from "how much deviation
should we forgive" to "how much should we trust this particular integer
call". Two independent confidence flags are added to every segment, neither
of which changes the match / over / under classification itself, they are
reported alongside it:

- **low_bin_count**: segments built from very few bins have a less reliable
  mean value, and a single aberrant bin can dominate a short segment's
  value. Segments with fewer bins than a minimum cutoff are flagged.
- **near_rounding_boundary**: Control-FREEC's CopyNumber is MedianRatio
  (the continuous, FREEC-smoothed signal) rounded to the nearest integer.
  A MedianRatio sitting close to a rounding boundary (for example, close to
  the midpoint between copy number 2 and 3) could easily round differently
  in different samples due to noise alone, even if the true underlying
  signal were identical. This is checked directly using each sample's own
  MedianRatio and CopyNumber values, without assuming any fixed or shared
  ratio-to-copy-number conversion, since parental and fusion samples sit at
  different baseline ploidies and Control-FREEC's subclone-aware calling
  does not reduce to one simple formula.

In [ ]:
def fit_ratio_to_cn_slope(seg_df, sample, medratio_col):
    """
    Fit CopyNumber ~ slope * MedianRatio through the origin for one sample,
    using that sample's own segments, weighted by segment size in base
    pairs so short, noisy segments do not dominate the fit. This recovers
    each sample's own ratio-to-copy-number scale (which depends on that
    sample's assumed ploidy) directly from data already on hand, without
    needing to know Control-FREEC's internal conversion.
    """
    x = seg_df[medratio_col].values
    y = seg_df[sample].values
    w = seg_df["segment_bp"].values
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0)
    x, y, w = x[valid], y[valid], w[valid]
    if len(x) == 0 or np.sum(w * x * x) == 0:
        return np.nan
    slope = np.sum(w * x * y) / np.sum(w * x * x)
    return slope

def add_segment_confidence_flags(seg_df, median_ratio_df, parent_a, parent_b, fusion,
                                 min_bins=3, boundary_threshold=0.15):
    """
    Add two confidence flags to every segment: low_bin_count (segment built
    from very few bins) and near_rounding_boundary (the underlying
    MedianRatio signal, for at least one of the three samples, sits close
    to the midpoint between two integer copy number calls).
    """
    out = seg_df.copy()
    out["low_bin_count"] = out["n_bins"] < min_bins

    samples = [parent_a, parent_b, fusion]

    # Attach mean MedianRatio per sample for each segment's genomic range.
    for sample in samples:
        medratio_col = f"{sample}__medratio"
        out[medratio_col] = np.nan
        sample_ratios = median_ratio_df[["Chromosome", "Start", sample]]
        for chrom, seg_group in out.groupby("Chromosome"):
            chrom_ratios = sample_ratios[sample_ratios["Chromosome"] == chrom]
            if chrom_ratios.empty:
                continue
            for idx, row in seg_group.iterrows():
                mask = (chrom_ratios["Start"] >= row["Start"]) & (chrom_ratios["Start"] <= row["End"])
                vals = chrom_ratios.loc[mask, sample]
                if len(vals) > 0:
                    out.at[idx, medratio_col] = vals.mean()

    distance_to_nearest_integer = pd.DataFrame(index=out.index)
    for sample in samples:
        medratio_col = f"{sample}__medratio"
        slope = fit_ratio_to_cn_slope(out, sample, medratio_col)
        predicted_continuous_cn = slope * out[medratio_col]
        distance_to_nearest_integer[sample] = (predicted_continuous_cn - predicted_continuous_cn.round()).abs()

    # distance_to_nearest_integer ranges from 0 (sitting exactly on a whole
    # number, most confident) to 0.5 (sitting exactly halfway between two
    # whole numbers, least confident). Convert to a distance FROM the
    # rounding boundary itself, so a small value means the call is risky.
    out["min_distance_from_boundary"] = (0.5 - distance_to_nearest_integer).min(axis=1)
    out["near_rounding_boundary"] = out["min_distance_from_boundary"] < boundary_threshold

    return out


### 7.1 Parental Subclone-Region Overlap (Low-Threshold Re-run)

Re-running Control-FREEC with `minimalSubclonePresence` lowered to 0.1
(from the main pipeline's 0.9) surfaced candidate subclonal structure in
several parents. The `_ratio.txt` files under `FREEC_BASE_DIR` have been
swapped in place with this low-threshold rerun's output (same files
merge_freec_cn_data already reads, now with two extra columns,
`Subclone_CN` and `Subclone_Population`, wherever FREEC called candidate
subclonal structure). This section adds a third confidence flag,
`subclone_region_overlap`, marking every segment where either parent has
a candidate subclone call overlapping its genomic range.

**This flag does not change match / over / under classification.** It is
reported alongside it, exactly like `low_bin_count` and
`near_rounding_boundary`.

**Named caveat, deliberately left unresolved:** because each fusion clone
arose from a single isolated fusion event, a non-additive fusion segment
that overlaps a flagged parental region is consistent with at least two
different explanations that look identical in bulk sequencing data --

1. a real copy-number event occurring after fusion, or
2. the fusion simply arising from a minor parental subclone rather than
   the major one, so the "parent" copy number used in the additive model
   (the major-clone call) was never the actual input to the fusion in
   that region.

This flag identifies *where* that ambiguity applies. It cannot and does
not attempt to adjudicate *which* of the two explanations is correct, and
no downstream code in this notebook should be read as resolving it either.


In [ ]:
def merge_freec_subclone_data(base_dir, allowed_chroms, samples):
    """
    Read Subclone_CN and Subclone_Population directly from the parental
    samples' _ratio.txt files under base_dir -- the same files, in the
    same flat directory layout, that merge_freec_cn_data already reads
    (base_dir/all_ratio_files*/*_ratio.txt). These are expected to now be
    the low-threshold (minimalSubclonePresence=0.1) rerun output, swapped
    in place of the original 0.9-threshold files.

    Only the samples passed in `samples` (the parental controls) are kept.
    Any other _ratio.txt files found under base_dir -- e.g. the fusion
    clones' own rerun output -- are ignored here, since this flag is
    parent-only.

    Each sample's own CopyNumber call is also kept (as f"{sample}__own_cn"),
    so gain/loss direction is derived from this same file's paired numbers.

    Only bins with Subclone_Population != 0 are informative. Subclone_CN
    can legitimately be 0 (homozygous deletion), so population, not CN, is
    the correct column to test for "nothing here."
    """
    ratio_files = glob.glob(os.path.join(base_dir, "all_ratio_files*", "*_ratio.txt"))
    if len(ratio_files) == 0:
        raise FileNotFoundError(f"No _ratio.txt files found under {base_dir}")

    samples_found = set()
    dfs = []
    for file in ratio_files:
        base = os.path.basename(file)
        sample_name = base.split("_paired")[0]
        if sample_name not in samples:
            continue
        samples_found.add(sample_name)

        df = pd.read_csv(file, sep="\t", dtype={"Chromosome": str})
        missing = {"Subclone_CN", "Subclone_Population"} - set(df.columns)
        if missing:
            raise ValueError(
                f"{sample_name}: _ratio.txt is missing expected column(s) {missing}. "
                f"Was this file actually overwritten with the lowered-minimalSubclonePresence rerun?"
            )

        df = df[df["Chromosome"].isin(allowed_chroms)]
        df = df.rename(columns={
            "CopyNumber": f"{sample_name}__own_cn",
            "Subclone_CN": f"{sample_name}__subclone_cn",
            "Subclone_Population": f"{sample_name}__subclone_population",
        })
        keep_cols = ["Chromosome", "Start", f"{sample_name}__own_cn",
                     f"{sample_name}__subclone_cn", f"{sample_name}__subclone_population"]
        dfs.append(df[keep_cols])

    missing_samples = set(samples) - samples_found
    if missing_samples:
        raise FileNotFoundError(f"No _ratio.txt file found under {base_dir} for: {sorted(missing_samples)}")

    merged_df = dfs[0]
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=["Chromosome", "Start"], how="inner")

    chrom_order = {chrom: i for i, chrom in enumerate(allowed_chroms)}
    merged_df["Chromosome_sort"] = merged_df["Chromosome"].map(chrom_order)
    merged_df = merged_df.sort_values(["Chromosome_sort", "Start"]).drop(columns="Chromosome_sort")
    merged_df = merged_df.reset_index(drop=True)

    print(f"Subclone data merged for {len(samples_found)} parental samples: {len(merged_df)} bins")
    return merged_df

subclone_bin_df = merge_freec_subclone_data(FREEC_BASE_DIR, ALLOWED_CHROMS, PARENTAL_SAMPLES)


In [ ]:
def add_subclone_overlap_flag(seg_df, subclone_bin_df, parent_a, parent_b):
    """
    Flag every segment where either parent has at least one bin, in the
    low-threshold (minimalSubclonePresence=0.1) re-run, with nonzero
    Subclone_Population overlapping the segment's genomic range.

    This does NOT change match / over / under classification -- it is
    reported alongside it, the same way low_bin_count and
    near_rounding_boundary are. See the Section 7.1 markdown for the named
    caveat this flag surfaces but does not resolve.

    Any-bp overlap is used (not a minimum overlap fraction) as a first
    pass -- intentionally permissive until we know how much this
    over-flags relative to the segments actually driving non-additive
    calls.

    subclone_overlap_detail lists, per parent with a hit, every distinct
    (subclone_CN, direction, population) call found among the overlapping
    bins, semicolon-joined -- a segment can straddle two different
    subclone fragments (or one flagged parent and one clean parent), so
    this is not collapsed down to a single value.
    """
    out = seg_df.copy()
    out["subclone_region_overlap"] = False
    out["subclone_overlap_detail"] = ""

    parent_role = {parent_a: "parent_a", parent_b: "parent_b"}

    for idx, row in out.iterrows():
        chrom_bins = subclone_bin_df[subclone_bin_df["Chromosome"] == row["Chromosome"]]
        if chrom_bins.empty:
            continue
        seg_bins = chrom_bins[(chrom_bins["Start"] >= row["Start"]) & (chrom_bins["Start"] <= row["End"])]
        if seg_bins.empty:
            continue

        details = []
        for parent in (parent_a, parent_b):
            pop_col = f"{parent}__subclone_population"
            cn_col = f"{parent}__subclone_cn"
            own_col = f"{parent}__own_cn"
            hits = seg_bins[seg_bins[pop_col] != 0]
            if hits.empty:
                continue

            unique_calls = hits[[cn_col, own_col, pop_col]].drop_duplicates()
            for _, call in unique_calls.iterrows():
                step = call[cn_col] - call[own_col]
                direction = "gain" if step > 0 else "loss"
                details.append(
                    f"{parent_role[parent]}:{parent},CN={int(call[cn_col])},{direction},pop={call[pop_col]:.4f}"
                )

        if details:
            out.at[idx, "subclone_region_overlap"] = True
            out.at[idx, "subclone_overlap_detail"] = "; ".join(details)

    return out


## 8. Continuous Copy-Number Residual (log2 Scale)

The `residual` column above is exact-integer, which is the right scale for
the match / over / under classification (Section 6), but not a fair
comparison against external log2 fold-change data (for example, RNA-seq
DESeq2 results contrasting a fusion clone against its mid-parent value): a
fixed CN residual represents a much larger proportional change in a
low-ploidy region than a high-ploidy one, while a log2 fold change is
already scale-normalized by construction. This section adds a continuous,
log2-scale companion metric, `log2_residual`, purely as an additional
column for that later correlation -- it does not replace, alter, or get
read by any of the exact-integer classification logic, confidence flags,
or plots above.

The reference point for `log2_residual` is deliberately kept as the same
additive sum (parent A continuous CN + parent B continuous CN) used
everywhere else in this notebook, not the geometric mean of the two
parents' individual log2 ratios -- the latter is a different, less
biologically justified null for DNA content, and would make this column
inconsistent with the additive model the rest of the notebook tests.

Requires `add_segment_confidence_flags` (Section 7) to have already run,
since it reuses the `f"{sample}__medratio"` columns and the
`fit_ratio_to_cn_slope` fit that section already provides.


In [ ]:
def add_continuous_log2_residual(seg_df, parent_a, parent_b, fusion, eps=0.5, cap=8.0):
    """
    Add a continuous, log2-scale companion to the exact-integer `residual`
    column (see the Section 8 markdown for why).

    Procedure, matching the additive model used everywhere else in this
    notebook:
      1. Recover each sample's own continuous CN-equivalent value at each
         segment as slope * MedianRatio, using that sample's own fitted
         ratio-to-CN slope (fit_ratio_to_cn_slope) -- avoids mixing samples
         that sit at different baseline ploidies onto one shared scale,
         and avoids the rounding-driven discretization the integer
         CopyNumber column already applies. Clipped at 0, since the linear
         fit has no non-negativity constraint and a noisy low-coverage
         segment could otherwise produce a small negative estimate.
      2. Sum the two parents' continuous values to get
         expected_additive_cont -- the continuous analog of the existing
         expected_additive column, using the same additive null (NOT the
         geometric mean of the two parents' log2 ratios).
      3. log2_residual = log2((fusion_cont + eps) / (expected_additive_cont + eps)),
         clipped to +/- cap.

    eps (default 0.5) is a pseudocount preventing log2(0) / log2(x/0) at
    segments where either side is at or near zero (e.g. biallelic
    deletions) -- the same role a pseudocount plays before log-transforming
    RNA-seq counts. At eps=0.5, the realistic range of fully-lost segments
    in this dataset (given observed parental CN roughly 0-60) stays within
    about +/-7, so cap=8 is a documented backstop that is not expected to
    routinely fire on real data -- it exists to bound genuinely pathological
    values, not to reshape the distribution under normal conditions. Both
    are function arguments so this can be revisited without editing the
    function body.

    A segment classified as an exact-integer `match` (residual == 0) will
    still generally have a nonzero log2_residual, since the continuous
    inputs carry real sub-integer variation that rounding discards -- that
    is expected, not a bug: this metric is meant to recover exactly the
    proportional information the integer model deliberately throws away.

    This is purely an additive companion metric: the exact-integer
    residual/classification columns already on seg_df are left untouched,
    and no plot or downstream function that reads them is affected.
    """
    out = seg_df.copy()
    samples = [parent_a, parent_b, fusion]

    cont_cols = {}
    for sample in samples:
        medratio_col = f"{sample}__medratio"
        if medratio_col not in out.columns:
            raise KeyError(f"{medratio_col} not found in seg_df -- run add_segment_confidence_flags first")
        slope = fit_ratio_to_cn_slope(out, sample, medratio_col)
        cont_col = f"{sample}__cn_cont"
        out[cont_col] = (slope * out[medratio_col]).clip(lower=0)
        cont_cols[sample] = cont_col

    out["expected_additive_cont"] = out[cont_cols[parent_a]] + out[cont_cols[parent_b]]

    fusion_cont = out[cont_cols[fusion]]
    expected_cont = out["expected_additive_cont"]
    raw_log2 = np.log2((fusion_cont + eps) / (expected_cont + eps))
    out["log2_residual"] = raw_log2.clip(lower=-cap, upper=cap)

    return out


## 9. Genome-Wide Quantification

The headline numbers for each fusion clone: the fraction of the genome
(base-pair weighted, and separately segment-count weighted) in each
classification, plus a single weighted MAE and a weighted mean signed
residual summarizing the average size and direction of deviation from
additivity.

In [ ]:
def compute_genome_wide_summary(seg_df):
    """
    Summarize the fraction of the genome in each classification, both by
    base pairs and by segment count, plus a genome-wide weighted MAE and
    mean signed residual.
    """
    total_bp = seg_df["segment_bp"].sum()
    total_segments = len(seg_df)

    rows = []
    for cls in ["match", "over", "under"]:
        sub = seg_df[seg_df["classification"] == cls]
        rows.append({
            "classification": cls,
            "n_segments": len(sub),
            "fraction_of_segments": len(sub) / total_segments if total_segments else np.nan,
            "bp": sub["segment_bp"].sum(),
            "fraction_of_bp": sub["segment_bp"].sum() / total_bp if total_bp else np.nan,
        })
    class_summary = pd.DataFrame(rows)

    weights = seg_df["segment_bp"].values
    resid = seg_df["residual"].values
    weighted_mae = np.average(np.abs(resid), weights=weights)
    weighted_mean_signed_resid = np.average(resid, weights=weights)

    overall_summary = pd.DataFrame([{
        "n_segments": total_segments,
        "total_bp": total_bp,
        "weighted_MAE": weighted_mae,
        "weighted_mean_signed_residual": weighted_mean_signed_resid,
    }])

    return class_summary, overall_summary


### 9.1 Sensitivity Check: Excluding Subclone-Flagged Regions

Recomputes the same genome-wide statistics as above, after dropping every
segment flagged `subclone_region_overlap`, so we can see how much of a
fusion clone's non-additive pattern depends on those regions versus holds
up without them. Requires `add_subclone_overlap_flag` to have been run
first; if it hasn't (e.g. `subclone_bin_df` was not supplied), this step
is skipped rather than failing.


In [ ]:
def compute_genome_wide_summary_excluding_flagged(seg_df, flag_col="subclone_region_overlap"):
    """
    Sensitivity check: recompute the same genome-wide summary statistics
    as compute_genome_wide_summary, after dropping every segment flagged
    by flag_col.

    Returns the same (class_summary, overall_summary) tuple as
    compute_genome_wide_summary, computed only on the unflagged subset,
    plus a one-row exclusion_report describing how much was dropped (by
    segment count and by bp) so the sensitivity of the result to the
    exclusion is visible alongside the recomputed numbers themselves.
    """
    if flag_col not in seg_df.columns:
        raise KeyError(f"'{flag_col}' not found in seg_df -- run add_subclone_overlap_flag first")

    total_bp = seg_df["segment_bp"].sum()
    total_segments = len(seg_df)

    excluded = seg_df[seg_df[flag_col]]
    kept = seg_df[~seg_df[flag_col]]

    exclusion_report = pd.DataFrame([{
        "n_segments_total": total_segments,
        "n_segments_excluded": len(excluded),
        "fraction_segments_excluded": len(excluded) / total_segments if total_segments else np.nan,
        "bp_excluded": excluded["segment_bp"].sum(),
        "fraction_bp_excluded": excluded["segment_bp"].sum() / total_bp if total_bp else np.nan,
    }])

    class_summary, overall_summary = compute_genome_wide_summary(kept)
    return class_summary, overall_summary, exclusion_report


## 10. Bootstrap Confidence Intervals

Segments within a chromosome are not independent, so confidence intervals
are built by resampling whole chromosomes with replacement (chromosome-block
bootstrap) rather than resampling individual segments.

In [ ]:
def bootstrap_genome_wide_summary(seg_df, n_boot=3000, random_seed=42):
    """
    Bootstrap confidence intervals for the bp-weighted classification
    fractions, the weighted MAE, and the weighted mean signed residual,
    resampling whole chromosomes with replacement.
    """
    rng = np.random.default_rng(random_seed)
    chrom_groups = {chrom: g.copy() for chrom, g in seg_df.groupby("Chromosome", sort=False)}
    chroms = list(chrom_groups.keys())

    boot_rows = []
    for _ in range(n_boot):
        sampled = rng.choice(chroms, size=len(chroms), replace=True)
        boot_df = pd.concat([chrom_groups[c] for c in sampled], ignore_index=True)

        total_bp = boot_df["segment_bp"].sum()
        frac_match = boot_df.loc[boot_df["classification"] == "match", "segment_bp"].sum() / total_bp
        frac_over = boot_df.loc[boot_df["classification"] == "over", "segment_bp"].sum() / total_bp
        frac_under = boot_df.loc[boot_df["classification"] == "under", "segment_bp"].sum() / total_bp
        weighted_mae = np.average(np.abs(boot_df["residual"]), weights=boot_df["segment_bp"])
        weighted_mean_signed_resid = np.average(boot_df["residual"], weights=boot_df["segment_bp"])

        boot_rows.append({
            "fraction_match_bp": frac_match,
            "fraction_over_bp": frac_over,
            "fraction_under_bp": frac_under,
            "weighted_MAE": weighted_mae,
            "weighted_mean_signed_residual": weighted_mean_signed_resid,
        })

    boot_df_out = pd.DataFrame(boot_rows)

    summary_rows = []
    for col in ["fraction_match_bp", "fraction_over_bp", "fraction_under_bp",
                "weighted_MAE", "weighted_mean_signed_residual"]:
        summary_rows.append({
            "metric": col,
            "mean": boot_df_out[col].mean(),
            "CI_2.5pct": boot_df_out[col].quantile(0.025),
            "median": boot_df_out[col].median(),
            "CI_97.5pct": boot_df_out[col].quantile(0.975),
        })
    boot_summary = pd.DataFrame(summary_rows)

    return boot_df_out, boot_summary


## 11. Test for a Systematic Genome-Wide Gain or Loss Bias

A Wilcoxon signed-rank test on segment residuals, testing whether they are
systematically shifted away from zero. This tests for an overall gain or
loss bias across the genome. It is not a test of which model best fits the
data, and each segment is treated as one observation.

In [ ]:
def test_residual_bias(seg_df):
    """
    Test whether segment residuals are systematically shifted away from
    zero genome-wide.
    """
    resid = seg_df["residual"].values
    resid = resid[np.isfinite(resid)]
    if len(resid) < 2 or np.all(resid == resid[0]):
        return pd.DataFrame([{
            "n_segments": len(resid),
            "median_residual": np.median(resid) if len(resid) else np.nan,
            "Statistic": np.nan,
            "P_value": np.nan,
            "note": "Not enough variation to test.",
        }])
    stat, p = wilcoxon(resid, alternative="two-sided", zero_method="wilcox")
    return pd.DataFrame([{
        "n_segments": len(resid),
        "median_residual": np.median(resid),
        "Statistic": stat,
        "P_value": p,
        "note": "",
    }])


## 12. Chromosome-Arm-Level Rollup

Rolls segment-level calls up to one call per chromosome arm, using a
base-pair-weighted majority vote for the classification and a
base-pair-weighted mean signed residual for that arm. This is the most
reviewer-legible summary unit: one row per arm per fusion clone.

In [ ]:
def build_arm_level_calls(seg_df):
    """
    Roll segment-level calls up to one call per chromosome arm, using a
    base-pair-weighted majority vote plus the weighted mean residual.
    """
    rows = []
    for arm, g in seg_df.groupby("Arm", sort=False):
        weights = g["segment_bp"].values
        bp_by_class = g.groupby("classification")["segment_bp"].sum()
        majority_class = bp_by_class.idxmax()
        weighted_resid = np.average(g["residual"].values, weights=weights)
        rows.append({
            "Arm": arm,
            "Chromosome": g["Chromosome"].iloc[0],
            "n_segments": len(g),
            "total_bp": weights.sum(),
            "majority_classification": majority_class,
            "weighted_mean_residual": weighted_resid,
        })
    arm_df = pd.DataFrame(rows)
    return arm_df


## 13. Confidence Flag Coverage by Arm

Before trusting an arm's call in the ideogram, it helps to know whether
that call rests on well-supported segments or is largely built from
segments already flagged low_bin_count or near_rounding_boundary. This
computes, per arm, the base-pair fraction coming from segments carrying
either flag, so a dramatic-looking arm can be checked directly: if most of
its bp comes from unflagged segments, the call is on solid footing; if
most comes from flagged segments, it deserves a closer look before being
treated as a firm result.

In [ ]:
def compute_arm_confidence_coverage(seg_df):
    """
    For each chromosome arm, the base-pair fraction coming from segments
    flagged low_bin_count or near_rounding_boundary (either flag), versus
    segments with neither flag.
    """
    out = seg_df.copy()
    out["any_flag"] = out["low_bin_count"] | out["near_rounding_boundary"]

    rows = []
    for arm, g in out.groupby("Arm", sort=False):
        total_bp = g["segment_bp"].sum()
        flagged_bp = g.loc[g["any_flag"], "segment_bp"].sum()
        rows.append({
            "Arm": arm,
            "total_bp": total_bp,
            "flagged_bp": flagged_bp,
            "fraction_flagged_bp": flagged_bp / total_bp if total_bp else np.nan,
        })
    coverage_df = pd.DataFrame(rows)
    arm_order = {arm: i for i, arm in enumerate(sort_arms(coverage_df["Arm"]))}
    coverage_df["_sort_key"] = coverage_df["Arm"].map(arm_order)
    coverage_df = coverage_df.sort_values("_sort_key").drop(columns="_sort_key").reset_index(drop=True)
    return coverage_df


## 14. Plotting Functions

The following figures are produced per fusion clone:

- Genome-wide segment track, colored by classification
- Observed versus expected scatter plot
- Residual histogram with the calibrated match tolerance marked
- Single-chromosome zoom-in view

And across all fusion clones together:

- Stacked bar chart of genome fraction in each classification
- Chromosome-arm-level ideogram (rows = clones, columns = arms)
- Recurrence plot (how many clones are non-additive at each arm)
- Residual heatmap across the whole genome for all clones

A single extreme value (a real focal amplification captured in one or two
bins, for example) can dominate a linear axis and squash every other,
biologically more typical point into unreadable clutter. The scatter and
residual plots below check their own data for this and switch to a symlog
axis (linear near zero, logarithmic beyond a data-driven threshold) only
when an extreme value is actually present, rather than always using it.

In [ ]:
def needs_symlog(values, factor=5, min_value_for_check=20):
    """
    Decide whether a symlog axis is needed based on how extreme the
    largest magnitude value is relative to the rest of the distribution.
    The reference percentile is computed after excluding the single
    largest value itself, so a severe outlier cannot inflate its own
    comparison baseline (this matters most at small segment counts, where
    one point can otherwise dominate a percentile of the full data).
    min_value_for_check avoids triggering symlog over small, low-magnitude
    data where this ratio is not a meaningful signal.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 5:
        return False
    abs_sorted = np.sort(np.abs(values))
    max_val = abs_sorted[-1]
    if max_val < min_value_for_check:
        return False
    rest = abs_sorted[:-1]
    reference = np.percentile(rest, 95) if len(rest) else 0.0
    if reference <= 0:
        reference = rest.max() if len(rest) else 0.0
    if reference <= 0:
        return False
    return max_val > factor * reference


In [ ]:
def plot_segment_track(seg_df, fusion, allowed_chroms, title=None, name=None, out_dir=None):
    """
    Draw every segment along the genome, colored by its classification
    relative to the additive expectation.
    """
    fusion_label = get_label(fusion)
    plot_df, chrom_centers, offsets = add_segment_genome_coords(seg_df, allowed_chroms)

    fig, ax = plt.subplots(figsize=(18, 4))
    for cls, color in CLASS_COLORS.items():
        sub = plot_df[plot_df["classification"] == cls]
        ax.hlines(sub[fusion], sub["genome_start"], sub["genome_end"], color=color, linewidth=4, label=cls)

    chrom_boundaries = sorted(offsets.values())[1:]
    for x in chrom_boundaries:
        ax.axvline(x, color="black", linewidth=0.5, alpha=0.3)

    ax.set_xticks([chrom_centers[c] for c in chrom_centers])
    ax.set_xticklabels(list(chrom_centers.keys()))
    ax.set_xlabel("Chromosome")
    ax.set_ylabel("Fusion copy number")
    ax.set_title(title or f"{fusion_label} segments colored by fit to additive model")
    ax.legend(loc="upper right")
    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()
    return plot_df


In [ ]:
def plot_observed_vs_expected_scatter(seg_df, fusion, title=None, name=None, out_dir=None):
    """
    Scatter of observed fusion copy number versus the expected additive
    copy number, colored by classification, with a y = x reference line.
    Point size encodes n_bins (how many bins support that segment's mean
    value), shown with its own separate size legend, since matplotlib does
    not scale legend swatches to match point size automatically. Switches
    to a symlog axis automatically if an extreme value is present.
    """
    fusion_label = get_label(fusion)
    fig, ax = plt.subplots(figsize=(6, 6))
    for cls, color in CLASS_COLORS.items():
        sub = seg_df[seg_df["classification"] == cls]
        ax.scatter(sub["expected_additive"], sub[fusion], s=sub["n_bins"] * 3 + 5,
                  color=color, alpha=0.6, edgecolor="none")

    combined_values = pd.concat([seg_df["expected_additive"], seg_df[fusion]])
    max_val = combined_values.max() + 1
    ax.plot([0, max_val], [0, max_val], color="black", linestyle="--", linewidth=1)

    if needs_symlog(combined_values):
        linthresh = max(10.0, float(np.percentile(np.abs(combined_values), 95)))
        ax.set_xscale("symlog", linthresh=linthresh)
        ax.set_yscale("symlog", linthresh=linthresh)
        print(f"{fusion}: using symlog axis scale (extreme value present, max {combined_values.max():.0f})")

    ax.set_xlabel("Expected additive copy number (parent A + parent B)")
    ax.set_ylabel("Observed fusion copy number")
    ax.set_title(title or f"{fusion_label}: observed versus expected additive copy number")

    # Color legend with fixed-size marker handles, so the swatches reflect
    # classification color only and are not accidentally sized by whichever
    # point matplotlib happened to grab from each scatter call.
    color_handles = [
        plt.Line2D([0], [0], marker="o", linestyle="none", color=color, markersize=8, label=cls)
        for cls, color in CLASS_COLORS.items()
    ]
    color_handles.append(plt.Line2D([0], [0], color="black", linestyle="--", label="y = x"))
    color_legend = ax.legend(handles=color_handles, loc="upper left", title="classification")
    ax.add_artist(color_legend)

    # Separate size legend for n_bins, using the same size formula as the
    # plotted points so the reference markers are directly comparable.
    size_reference_bins = [1, 10, 50]
    size_handles = [
        ax.scatter([], [], s=n * 3 + 5, color="0.5", alpha=0.6, edgecolor="none", label=str(n))
        for n in size_reference_bins
    ]
    ax.legend(handles=size_handles, loc="lower right", title="n_bins in segment")

    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()


In [ ]:
def plot_residual_histogram(seg_df, title=None, name=None, out_dir=None):
    """
    Bar chart of residuals over their exact integer values. Residuals are
    always exact integers (see section 6), so a continuous histogram with
    arbitrary bin edges would misrepresent the data. A discrete bar chart
    over the integer values is the correct representation here. Switches
    to a symlog x-axis automatically if an extreme residual is present.
    """
    counts = seg_df["residual"].astype(int).value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(counts.index, counts.values, color="0.5", width=0.8)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Residual (fusion minus expected additive, exact copies)")
    ax.set_ylabel("Number of segments")

    if needs_symlog(counts.index.values):
        linthresh = max(5.0, float(np.percentile(np.abs(counts.index.values), 95)))
        ax.set_xscale("symlog", linthresh=linthresh)
        print(f"Using symlog x-axis scale (extreme residual present, max magnitude {np.abs(counts.index.values).max()})")
    else:
        ax.set_xticks(list(counts.index))

    ax.set_title(title or "Distribution of residuals from the additive expectation")
    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()


In [ ]:
def plot_log2_residual_vs_integer_residual(seg_df, fusion, title=None, name=None, out_dir=None):
    """
    Diagnostic scatter of the continuous log2_residual (Section 8) against
    the original exact-integer residual, colored by classification, with
    point size encoding expected_additive_cont (the segment's baseline
    copy-number scale). This is a sanity check on the log-ratio version,
    not a replacement for the integer classification: segments sharing the
    same integer residual should show a spread of log2_residual values
    that tracks baseline ploidy (the same absolute CN step is a bigger
    proportional change in a low-copy region than a high-copy one), rather
    than collapsing to one value per integer residual.
    """
    fusion_label = get_label(fusion)
    fig, ax = plt.subplots(figsize=(7, 6))
    rng = np.random.default_rng(0)
    jitter = pd.Series(rng.uniform(-0.12, 0.12, size=len(seg_df)), index=seg_df.index)
    sizes = 15 + 4 * np.sqrt(seg_df["expected_additive_cont"].clip(lower=0))

    for cls, color in CLASS_COLORS.items():
        sub = seg_df[seg_df["classification"] == cls]
        ax.scatter(sub["residual"] + jitter.loc[sub.index], sub["log2_residual"],
                   s=sizes.loc[sub.index], color=color, alpha=0.6, edgecolor="none", label=cls)

    ax.axhline(0, color="black", linewidth=0.8, alpha=0.5)
    ax.axvline(0, color="black", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("Exact-integer residual (fusion minus expected additive; jittered for visibility)")
    ax.set_ylabel("log2_residual (continuous)")
    ax.set_title(title or f"{fusion_label}: continuous log2 residual vs. exact-integer residual")
    ax.legend(title="classification", loc="best")
    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()


In [ ]:
def plot_classification_stacked_bar(class_summaries, name=None, out_dir=None,figsize=(6, 5)):
    """
    class_summaries: dict mapping fusion clone name to its class_summary
    dataframe (output of compute_genome_wide_summary).
    """
    clones = list(class_summaries.keys())
    clone_labels = [get_label(c) for c in clones]
    fractions = {cls: [] for cls in ["match", "over", "under"]}
    for clone in clones:
        cs = class_summaries[clone].set_index("classification")
        for cls in fractions:
            fractions[cls].append(cs.loc[cls, "fraction_of_bp"] if cls in cs.index else 0.0)

    fig, ax = plt.subplots(figsize=figsize)
    bottom = np.zeros(len(clones))
    for cls in ["match", "over", "under"]:
        vals = np.array(fractions[cls]) * 100
        ax.bar(clone_labels, vals, bottom=bottom, color=CLASS_COLORS[cls], label=cls)
        bottom += vals

    ax.set_ylabel("Percent of genome\n(base pair weighted)")
    ax.set_title("Fraction of genome matching, above, \nor below additive expectation")
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=FONT_SIZE-2)
    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()


In [ ]:
def plot_arm_ideogram(arm_level_by_clone, name=None, out_dir=None):
    """
    arm_level_by_clone: dict mapping fusion clone name to its arm-level
    dataframe (output of build_arm_level_calls).
    """
    clones = list(arm_level_by_clone.keys())
    all_arms = sort_arms(set().union(*[set(df["Arm"]) for df in arm_level_by_clone.values()]))

    class_to_num = {"match": 0, "over": 1, "under": -1}
    matrix = np.full((len(clones), len(all_arms)), np.nan)
    for i, clone in enumerate(clones):
        df = arm_level_by_clone[clone].set_index("Arm")
        for j, arm in enumerate(all_arms):
            if arm in df.index:
                matrix[i, j] = class_to_num.get(df.loc[arm, "majority_classification"], np.nan)

    cmap = mcolors.ListedColormap(["steelblue", "0.85", "firebrick"])
    bounds = [-1.5, -0.5, 0.5, 1.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    fig, ax = plt.subplots(figsize=(max(12, len(all_arms) * 0.3), 1 + len(clones) * 0.9))
    ax.imshow(matrix, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_yticks(range(len(clones)))
    ax.set_yticklabels([get_label(c) for c in clones])
    ax.set_xticks(range(len(all_arms)))
    ax.set_xticklabels(all_arms, rotation=90)
    ax.set_title("Chromosome arm classification per fusion clone")

    legend_patches = [
        mpatches.Patch(color="firebrick", label="over"),
        mpatches.Patch(color="0.85", label="match"),
        mpatches.Patch(color="steelblue", label="under"),
    ]
    ax.legend(handles=legend_patches, loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3)
    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()
    return all_arms, matrix


In [ ]:
def plot_recurrence(arm_level_by_clone, name=None, out_dir=None):
    """
    Count, for each chromosome arm, how many fusion clones show a
    non-match (over or under) classification there.
    """
    all_arms = sort_arms(set().union(*[set(df["Arm"]) for df in arm_level_by_clone.values()]))
    counts = []
    for arm in all_arms:
        n = 0
        for clone, df in arm_level_by_clone.items():
            row = df[df["Arm"] == arm]
            if len(row) and row["majority_classification"].iloc[0] != "match":
                n += 1
        counts.append(n)

    fig, ax = plt.subplots(figsize=(max(12, len(all_arms) * 0.3), 4))
    ax.bar(all_arms, counts, color="0.4")
    ax.set_xticks(range(len(all_arms)))
    ax.set_xticklabels(all_arms, rotation=90)
    ax.set_ylabel("Number of fusion clones non-additive at this arm")
    ax.set_title("Recurrence of non-additive calls across fusion clones")
    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()

    return pd.DataFrame({"Arm": all_arms, "n_clones_non_additive": counts})


In [ ]:
def plot_single_chromosome_zoom(seg_df, parent_a, parent_b, fusion, chromosome, centromeres, name=None, out_dir=None):
    """
    Segment-level view of a single chromosome, showing the fusion,
    the additive expectation, and both parents, with the centromere marked.
    """
    sub = seg_df[seg_df["Chromosome"] == str(chromosome)].copy()
    if len(sub) == 0:
        raise ValueError(f"No segments found for chromosome {chromosome}")

    fusion_label = get_label(fusion)
    parent_a_label = get_label(parent_a)
    parent_b_label = get_label(parent_b)
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.hlines(sub[fusion], sub["Start"], sub["End"], color="black", linewidth=2, label=fusion_label)
    ax.hlines(sub["expected_additive"], sub["Start"], sub["End"], color="firebrick", linewidth=2,
             linestyle="--", label=f"{parent_a_label} + {parent_b_label}")
    ax.hlines(sub[parent_a], sub["Start"], sub["End"], color="royalblue", linewidth=1, alpha=0.6, label=parent_a_label)
    ax.hlines(sub[parent_b], sub["Start"], sub["End"], color="darkorange", linewidth=1, alpha=0.6, label=parent_b_label)

    cen = centromeres.get(str(chromosome))
    if cen is not None:
        ax.axvline(cen, color="gray", linewidth=1, linestyle=":", label="centromere")

    ax.set_xlabel(f"Chromosome {chromosome} position")
    ax.set_ylabel("Copy number")
    ax.set_title(f"{fusion_label}: chromosome {chromosome} segment-level view")
    ax.legend()
    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()
    return sub


In [ ]:
def build_residual_heatmap_matrix(bin_df, fusion_pairs, allowed_chroms, superbin_size=5_000_000):
    """
    Build a clone-by-genome-position matrix of mean (fusion minus additive
    expectation) residuals at a coarser super-bin resolution, for visual
    comparison of where each fusion clone deviates from additivity.
    """
    df, chrom_centers = add_genome_x(bin_df, allowed_chroms)
    df["superbin"] = (df["genome_x"] // superbin_size).astype(int)

    rows = []
    for fusion, (parent_a, parent_b) in fusion_pairs.items():
        resid = df[fusion] - (df[parent_a] + df[parent_b])
        tmp = pd.DataFrame({"superbin": df["superbin"], "residual": resid})
        grouped = tmp.groupby("superbin")["residual"].mean()
        rows.append(grouped.rename(fusion))

    matrix_df = pd.concat(rows, axis=1).T
    matrix_df = matrix_df.sort_index(axis=1)
    return matrix_df, chrom_centers, superbin_size

def plot_residual_heatmap(matrix_df, chrom_centers, superbin_size, name=None, out_dir=None):
    fig, ax = plt.subplots(figsize=(18, 1 + 0.6 * len(matrix_df)))
    im = ax.imshow(matrix_df.values, aspect="auto", cmap="coolwarm", vmin=-2, vmax=2, interpolation="nearest")
    ax.set_yticks(range(len(matrix_df.index)))
    ax.set_yticklabels([get_label(s) for s in matrix_df.index])

    col_list = list(matrix_df.columns)
    tick_positions = []
    tick_labels = []
    for chrom, center in chrom_centers.items():
        target = center / superbin_size
        idx = min(range(len(col_list)), key=lambda i: abs(col_list[i] - target))
        tick_positions.append(idx)
        tick_labels.append(chrom)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)

    cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    cbar.ax.tick_params(labelsize=FONT_SIZE - 2)
    cbar.set_label("Mean residual (fusion minus additive)")
    ax.set_title("Residual from additive expectation across the genome, all fusion clones")
    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()


## 15. Extreme Segment Diagnostic

A segment with a very large expected additive value could reflect a real
sustained amplification, or could be an artifact where a segment built
from very few bins has its mean value dragged around by a single aberrant
bin. This displays the most extreme segments by a chosen column (expected
additive copy number by default) along with the raw per-bin CopyNumber
values underneath them for both parents and the fusion, so each one can be
checked directly: a real amplification should show a high value sustained
across many consecutive bins, not concentrated in just one or two.

In [ ]:
def show_extreme_segments(seg_df, bin_df, parent_a, parent_b, fusion, n=10,
                          sort_by="expected_additive", output_tsv=None):
    """
    Display the most extreme segments by sort_by (expected_additive by
    default), along with the raw per-bin CopyNumber values underneath them,
    to help distinguish a real sustained signal from a low-bin-count
    artifact.
    """
    top = seg_df.sort_values(sort_by, ascending=False).head(n).copy()

    for _, row in top.iterrows():
        print("-" * 60)
        print(f"Chromosome {row['Chromosome']} ({row['Arm']}), "
             f"Start {row['Start']}, End {row['End']}, n_bins {row['n_bins']}")
        print(f"{parent_a} = {row[parent_a]}, {parent_b} = {row[parent_b]}, "
             f"{fusion} = {row[fusion]}, expected_additive = {row['expected_additive']}, "
             f"residual = {row['residual']}")
        sub_bins = bin_df[
            (bin_df["Chromosome"] == row["Chromosome"]) &
            (bin_df["Start"] >= row["Start"]) &
            (bin_df["Start"] <= row["End"])
        ]
        display(sub_bins[["Chromosome", "Start", parent_a, parent_b, fusion]])

    if output_tsv is not None:
        top.to_csv(output_tsv, sep="\t", index=False)

    return top


## 16. Master Runner

Ties every step above together for one fusion clone and its two parental
clones, saving all intermediate tables and figures to a dedicated output
directory.

In [ ]:
def run_fusion_pair_analysis(cn_df, median_ratio_df, parent_a, parent_b, fusion, output_dir, centromeres,
                             allowed_chroms=ALLOWED_CHROMS, min_bins=3, boundary_threshold=0.15,
                             n_boot=3000, random_seed=42, chromosome_to_plot="1", n_extreme=10,
                             subclone_bin_df=None, log2_residual_eps=0.5, log2_residual_cap=8.0):
    """
    Run the full arm-aware, segment-level additive inheritance analysis
    for one fusion clone and its two parental clones.
    """
    os.makedirs(output_dir, exist_ok=True)
    results = {}

    print("=" * 90)
    print(f"{fusion}: building arm-aware segments")
    print("=" * 90)
    seg_df = build_arm_aware_segments(
        cn_df, parent_a, parent_b, fusion, centromeres, allowed_chroms,
        output_tsv=os.path.join(output_dir, "01_arm_aware_segments.tsv")
    )
    seg_df = add_expected_models(seg_df, parent_a, parent_b, fusion)
    print(f"Segments built: {len(seg_df)}")
    results["segments_raw"] = seg_df

    print()
    print("=" * 90)
    print(f"{fusion}: classifying segments")
    print("=" * 90)
    seg_df = classify_segments(seg_df, parent_a, parent_b, fusion)
    print(seg_df["classification"].value_counts().to_string())

    print()
    print("=" * 90)
    print(f"{fusion}: adding segment confidence flags")
    print("=" * 90)
    seg_df = add_segment_confidence_flags(seg_df, median_ratio_df, parent_a, parent_b, fusion,
                                          min_bins=min_bins, boundary_threshold=boundary_threshold)
    print(f"Segments flagged low_bin_count (fewer than {min_bins} bins): {int(seg_df['low_bin_count'].sum())} of {len(seg_df)}")
    print(f"Segments flagged near_rounding_boundary: {int(seg_df['near_rounding_boundary'].sum())} of {len(seg_df)}")

    if subclone_bin_df is not None:
        seg_df = add_subclone_overlap_flag(seg_df, subclone_bin_df, parent_a, parent_b)
        print(f"Segments overlapping a parental subclone region (low-threshold re-run): "
              f"{int(seg_df['subclone_region_overlap'].sum())} of {len(seg_df)}")
    else:
        print("No subclone_bin_df supplied -- skipping subclone_region_overlap flag.")

    print()
    print("=" * 90)
    print(f"{fusion}: continuous (non-rounded) log2 residual")
    print("=" * 90)
    seg_df = add_continuous_log2_residual(seg_df, parent_a, parent_b, fusion,
                                          eps=log2_residual_eps, cap=log2_residual_cap)
    print(f"log2_residual added (eps={log2_residual_eps}, cap=+/-{log2_residual_cap}). "
          f"Observed range: [{seg_df['log2_residual'].min():.2f}, {seg_df['log2_residual'].max():.2f}]")

    seg_df.to_csv(os.path.join(output_dir, "02_classified_segments.tsv"), sep="\t", index=False)
    results["segments_classified"] = seg_df

    print()
    print("=" * 90)
    print(f"{fusion}: genome-wide quantification")
    print("=" * 90)
    class_summary, overall_summary = compute_genome_wide_summary(seg_df)
    display(class_summary)
    display(overall_summary)
    class_summary.to_csv(os.path.join(output_dir, "03_classification_summary.tsv"), sep="\t", index=False)
    overall_summary.to_csv(os.path.join(output_dir, "03_overall_summary.tsv"), sep="\t", index=False)
    results["class_summary"] = class_summary
    results["overall_summary"] = overall_summary

    if "subclone_region_overlap" in seg_df.columns:
        print()
        print("=" * 90)
        print(f"{fusion}: genome-wide quantification excluding subclone-flagged regions (sensitivity check)")
        print("=" * 90)
        class_summary_excl, overall_summary_excl, subclone_exclusion_report = \
            compute_genome_wide_summary_excluding_flagged(seg_df)
        display(subclone_exclusion_report)
        display(class_summary_excl)
        display(overall_summary_excl)
        subclone_exclusion_report.to_csv(
            os.path.join(output_dir, "03b_subclone_exclusion_report.tsv"), sep="\t", index=False)
        class_summary_excl.to_csv(
            os.path.join(output_dir, "03b_classification_summary_excl_subclone.tsv"), sep="\t", index=False)
        overall_summary_excl.to_csv(
            os.path.join(output_dir, "03b_overall_summary_excl_subclone.tsv"), sep="\t", index=False)
        results["class_summary_excl_subclone"] = class_summary_excl
        results["overall_summary_excl_subclone"] = overall_summary_excl
        results["subclone_exclusion_report"] = subclone_exclusion_report

    print()
    print("=" * 90)
    print(f"{fusion}: bootstrap confidence intervals")
    print("=" * 90)
    boot_raw, boot_summary = bootstrap_genome_wide_summary(seg_df, n_boot=n_boot, random_seed=random_seed)
    display(boot_summary)
    boot_summary.to_csv(os.path.join(output_dir, "04_bootstrap_summary.tsv"), sep="\t", index=False)
    results["bootstrap_summary"] = boot_summary

    print()
    print("=" * 90)
    print(f"{fusion}: testing for a systematic gain or loss bias")
    print("=" * 90)
    bias_test = test_residual_bias(seg_df)
    display(bias_test)
    bias_test.to_csv(os.path.join(output_dir, "05_residual_bias_test.tsv"), sep="\t", index=False)
    results["bias_test"] = bias_test

    print()
    print("=" * 90)
    print(f"{fusion}: chromosome-arm-level rollup")
    print("=" * 90)
    arm_df = build_arm_level_calls(seg_df)
    display(arm_df)
    arm_df.to_csv(os.path.join(output_dir, "07_arm_level_calls.tsv"), sep="\t", index=False)
    results["arm_level_calls"] = arm_df

    print()
    print("=" * 90)
    print(f"{fusion}: confidence flag coverage by arm")
    print("=" * 90)
    arm_confidence_coverage = compute_arm_confidence_coverage(seg_df)
    display(arm_confidence_coverage)
    arm_confidence_coverage.to_csv(os.path.join(output_dir, "08_arm_confidence_coverage.tsv"), sep="\t", index=False)
    results["arm_confidence_coverage"] = arm_confidence_coverage

    print()
    print("=" * 90)
    print(f"{fusion}: plots")
    print("=" * 90)
    plot_segment_track(seg_df, fusion, allowed_chroms,
                       name="09_segment_track", out_dir=output_dir)
    plot_observed_vs_expected_scatter(seg_df, fusion,
                                      name="10_observed_vs_expected", out_dir=output_dir)
    plot_residual_histogram(seg_df,
                            name="11_residual_histogram", out_dir=output_dir)
    if "log2_residual" in seg_df.columns:
        plot_log2_residual_vs_integer_residual(
            seg_df, fusion, name="11b_log2_residual_diagnostic", out_dir=output_dir)
    plot_single_chromosome_zoom(seg_df, parent_a, parent_b, fusion, chromosome_to_plot, centromeres,
                                name=f"12_chromosome_{chromosome_to_plot}_zoom", out_dir=output_dir)

    # print()
    # print("=" * 90)
    # print(f"{fusion}: extreme segment diagnostic (top {n_extreme} by expected additive copy number)")
    # print("=" * 90)
    # extreme_segments = show_extreme_segments(
    #     seg_df, cn_df, parent_a, parent_b, fusion, n=n_extreme,
    #     output_tsv=os.path.join(output_dir, "13_extreme_segments.tsv")
    # )
    # results["extreme_segments"] = extreme_segments

    # print()
    # print("=" * 90)
    # print(f"All outputs for {fusion} saved to {output_dir}")
    # print("=" * 90)

    return results


---
# Part 1 - Run the Analysis for Each Fusion Clone

In [ ]:
all_results = {}
for fusion, (parent_a, parent_b) in FUSION_PAIRS.items():
    all_results[fusion] = run_fusion_pair_analysis(
        cn_df, median_ratio_df,
        parent_a=parent_a, parent_b=parent_b, fusion=fusion,
        output_dir=os.path.join(BASE_OUTPUT_DIR, f"{fusion}_arm_level_outputs"),
        centromeres=centromeres, subclone_bin_df=subclone_bin_df,
    )

---
# Part 2 - Cross-Clone Comparison

Combines the three fusion clones into one set of comparison figures and
tables. Cross-clone differences are reported descriptively (bar chart with
bootstrap CIs, recurrence counts) rather than with a formal significance
test, since three biological fusion events is not enough to power a
between-clone hypothesis test without overstating the result.

### 2.H Helper Functions

In [ ]:
def chrom_sort_key(chrom):
    c = str(chrom).replace("chr", "")
    if c.isdigit():
        return int(c)
    return 23 if c.upper() == "X" else 24

def sort_df_by_genome(df):
    df = df.copy()
    df["_cs"] = df["Chromosome"].apply(chrom_sort_key)
    return df.sort_values(["_cs", "Start"]).drop(columns="_cs").reset_index(drop=True)

def sort_sample_cols(sample_cols):
    """Sort columns in SAMPLE_GROUP_ORDER, then alphabetically by display label."""
    def sort_key(s):
        grp = get_group(s)
        grp_idx = SAMPLE_GROUP_ORDER.index(grp) if grp in SAMPLE_GROUP_ORDER else 99
        return (grp_idx, get_label(s), s)
    return sorted(sample_cols, key=sort_key)

def merge_freec_ratio_data(base_dir, allowed_chroms, sample_names, retained_bins_df):
    """
    Merge the raw Ratio column from Control-FREEC's _ratio.txt files for
    sample_names, restricted to the bins already retained in cn_df
    (retained_bins_df -- already -1-filtered, blacklist-filtered, and
    MedianRatio-sentinel-corrected), then log2-transform. Ratio is not
    otherwise merged anywhere in the additive-inheritance analysis above,
    which only uses CopyNumber and MedianRatio.
    """
    ratio_files = glob.glob(os.path.join(base_dir, "all_ratio_files*", "*_ratio.txt"))
    by_sample = {}
    for f in ratio_files:
        sample_name = os.path.basename(f).split("_paired")[0]
        if sample_name in sample_names:
            by_sample[sample_name] = f

    missing = [s for s in sample_names if s not in by_sample]
    if missing:
        raise FileNotFoundError(f"No _ratio.txt found for: {missing}")

    dfs = []
    for sample in sample_names:
        df = pd.read_csv(by_sample[sample], sep="\t",
                         usecols=["Chromosome", "Start", "Ratio"],
                         dtype={"Chromosome": str})
        df = df[df["Chromosome"].isin(allowed_chroms)]
        df = df.rename(columns={"Ratio": sample})
        dfs.append(df)

    merged = dfs[0]
    for df in dfs[1:]:
        merged = pd.merge(merged, df, on=["Chromosome", "Start"], how="inner")

    # Drop FREEC's -1 sentinel before log2, same convention as the CopyNumber filter
    mask_bad = (merged[sample_names] == -1).any(axis=1)
    merged = merged[~mask_bad].copy()

    # Restrict to the exact bins already retained in cn_df, so this matrix
    # lines up bin-for-bin with the integer-CN additive analysis above
    keep_bins = retained_bins_df[["Chromosome", "Start"]]
    merged = pd.merge(merged, keep_bins, on=["Chromosome", "Start"], how="inner")

    merged[sample_names] = np.log2(merged[sample_names].replace(0, np.nan))
    merged = sort_df_by_genome(merged)
    return merged


def plot_chromosome_heatmap_log2(matrix_df, chromosome, sample_cols,
                                 value_label=r"log$_2$ CNR", vmin=-1.0, vmax=1.0,
                                 cmap="RdBu_r", output_prefix="chr_log2cnr"):
    chrom_str = str(chromosome).replace("chr", "")
    df_chr = matrix_df[matrix_df["Chromosome"].apply(lambda c: str(c).replace("chr", "")) == chrom_str].copy()
    if df_chr.empty:
        print(f"No data for chromosome {chrom_str}")
        return

    df_chr = sort_df_by_genome(df_chr).reset_index(drop=True)
    n = len(sample_cols)
    fig, axes = plt.subplots(n, 1, figsize=(14, max(3, 0.65 * n + 1.2)), squeeze=False)
    fig.subplots_adjust(hspace=0.05, right=0.86)

    for i, s in enumerate(sample_cols):
        ax = axes[i, 0]
        img = np.array([df_chr[s].to_numpy(dtype=float)])
        ax.imshow(img, aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap, interpolation="nearest")
        ax.set_yticks([]); ax.set_xticks([])
        color = GROUP_COLORS.get(get_group(s), "gray")
        ax.set_ylabel(get_label(s), rotation=0,
                      ha="right", va="center", labelpad=4,
                      color=color, fontweight="bold")
        for sp in ax.spines.values():
            sp.set_linewidth(0.6); sp.set_edgecolor(color)

    cbar_ax = fig.add_axes([0.88, 0.12, 0.025, 0.76])
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax)),
                        cax=cbar_ax, label=value_label)
    cbar.ax.tick_params(labelsize=FONT_SIZE - 2)
    fig.suptitle(f"Chromosome {chrom_str} - {value_label}", fontweight="bold", y=1.01)
    save_fig(fig, f"{output_prefix}_chr{chrom_str}")
    plt.show()

### 2.0 Main results

In [ ]:
# all_results = {
#     "C2C4_1806": c2c4_results,
#     "C5C2_1806": c5c2_results,
#     "C6C7_1806": c6c7_results,
# }

class_summaries = {clone: res["class_summary"] for clone, res in all_results.items()}
arm_level_by_clone = {clone: res["arm_level_calls"] for clone, res in all_results.items()}

cross_clone_dir = os.path.join(BASE_OUTPUT_DIR, "cross_clone_summary")
os.makedirs(cross_clone_dir, exist_ok=True)

plot_classification_stacked_bar(class_summaries, figsize=(8,6.25),
                                name="stacked_bar_classification", out_dir=cross_clone_dir)

all_arms, arm_matrix = plot_arm_ideogram(arm_level_by_clone,
                                         name="arm_ideogram", out_dir=cross_clone_dir)

recurrence_df = plot_recurrence(arm_level_by_clone,
                                name="recurrence", out_dir=cross_clone_dir)
display(recurrence_df.sort_values("n_clones_non_additive", ascending=False).head(20))
recurrence_df.to_csv(os.path.join(cross_clone_dir, "recurrence_table.tsv"), sep="\t", index=False)

overall_summary_table = pd.concat(
    [res["overall_summary"].assign(fusion=clone) for clone, res in all_results.items()],
    ignore_index=True
)
display(overall_summary_table)
overall_summary_table.to_csv(os.path.join(cross_clone_dir, "cross_clone_overall_summary.tsv"), sep="\t", index=False)


### 2.1 Residual Heatmap Across All Clones

In [ ]:
residual_matrix, chrom_centers, superbin_size = build_residual_heatmap_matrix(cn_df, FUSION_PAIRS, ALLOWED_CHROMS)
plot_residual_heatmap(residual_matrix, chrom_centers, superbin_size,
                      name="residual_heatmap_all_clones", out_dir=cross_clone_dir)


In [ ]:
def map_bins_to_segment_values(bin_df, seg_df, value_col, allowed_chroms):
    """
    Broadcast a segment-level value (e.g. log2_residual) back onto every
    bin the segment covers, using pd.merge_asof per chromosome. Segments
    from build_arm_aware_segments tile every retained bin exactly once and
    are contiguous and Start-sorted within a chromosome, so a backward
    as-of match on Start correctly assigns each bin to its containing
    segment -- no bin is left unmatched.
    """
    out_parts = []
    for chrom in allowed_chroms:
        bins_c = bin_df[bin_df["Chromosome"] == chrom][["Chromosome", "Start"]].sort_values("Start")
        segs_c = seg_df[seg_df["Chromosome"] == chrom][["Start", value_col]].sort_values("Start")
        if bins_c.empty or segs_c.empty:
            continue
        merged = pd.merge_asof(bins_c, segs_c, on="Start", direction="backward")
        out_parts.append(merged)
    return pd.concat(out_parts, ignore_index=True)[["Chromosome", "Start", value_col]]


def build_log2_residual_heatmap_matrix(cn_df, all_results, fusion_pairs, allowed_chroms, superbin_size=5_000_000):
    """
    Clone-by-genome-position matrix of mean log2_residual (Section 8) at
    superbin resolution, parallel to build_residual_heatmap_matrix but in
    log2 space. Unlike the integer-residual version -- computed fresh from
    raw bin-level CopyNumber, since that needs no model fit -- this reuses
    each fusion pair's own already-computed segment-level log2_residual,
    broadcasting each segment's single value onto every bin it covers,
    rather than independently refitting a bin-level slope here. Requires
    all_results[fusion]["segments_classified"] to already carry
    log2_residual for every pair (true after run_fusion_pair_analysis has
    been run, since Section 8 always computes it).
    """
    df, chrom_centers = add_genome_x(cn_df, allowed_chroms)
    df["superbin"] = (df["genome_x"] // superbin_size).astype(int)

    rows = []
    for fusion in fusion_pairs:
        seg_df = all_results[fusion]["segments_classified"]
        if "log2_residual" not in seg_df.columns:
            raise KeyError(f"{fusion}: segments_classified has no log2_residual column -- "
                            f"was add_continuous_log2_residual run for this pair?")
        bin_values = map_bins_to_segment_values(df, seg_df, "log2_residual", allowed_chroms)
        tmp = df[["Chromosome", "Start", "superbin"]].merge(bin_values, on=["Chromosome", "Start"], how="left")
        rows.append(tmp.groupby("superbin")["log2_residual"].mean().rename(fusion))

    matrix_df = pd.concat(rows, axis=1).T
    matrix_df = matrix_df.sort_index(axis=1)
    return matrix_df, chrom_centers, superbin_size


def plot_log2_residual_heatmap(matrix_df, chrom_centers, superbin_size, vmax=None, name=None, out_dir=None):
    """
    Same layout as plot_residual_heatmap, in log2 space. vmax defaults to
    the 99th percentile of |matrix_df| (rather than log2_residual's own
    +/-8 cap from Section 8), since that cap is a rarely-hit backstop, not
    the range most segments actually occupy -- using it directly here
    would wash out real signal into a narrow band near white.
    """
    if vmax is None:
        vmax = max(0.5, float(np.ceil(np.nanpercentile(np.abs(matrix_df.values), 99) * 10) / 10))

    fig, ax = plt.subplots(figsize=(18, 1 + 0.6 * len(matrix_df)))
    im = ax.imshow(matrix_df.values, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax, interpolation="nearest")
    ax.set_yticks(range(len(matrix_df.index)))
    ax.set_yticklabels(matrix_df.index)

    col_list = list(matrix_df.columns)
    tick_positions, tick_labels = [], []
    for chrom, center in chrom_centers.items():
        target = center / superbin_size
        idx = min(range(len(col_list)), key=lambda i: abs(col_list[i] - target))
        tick_positions.append(idx)
        tick_labels.append(chrom)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)

    cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    cbar.ax.tick_params(labelsize=FONT_SIZE - 2)
    cbar.set_label("Mean log2_residual (fusion vs. additive expectation)")
    ax.set_title("log2-scale residual from additive expectation across the genome, all fusion clones")
    fig.tight_layout()

    if name:
        save_fig(fig, name, out_dir=out_dir)
    plt.show()


log2_residual_matrix, chrom_centers_log2, superbin_size_log2 = build_log2_residual_heatmap_matrix(
    cn_df, all_results, FUSION_PAIRS, ALLOWED_CHROMS
)
plot_log2_residual_heatmap(log2_residual_matrix, chrom_centers_log2, superbin_size_log2,
                           name="log2_residual_heatmap_all_clones", out_dir=cross_clone_dir)

### 2.2 Hierarchical Clustering

Clusters all 8 samples (parental controls and fusion clones) on their
genome-wide copy number profiles, to check whether each fusion clone
groups with its own two parents. Correlation distance is used rather than
Euclidean, since parental and fusion samples sit at very different
baseline ploidies (Section 3.4): correlation distance clusters on the
pattern of relative gains and losses along the genome, not on absolute
copy number level, so ploidy differences alone cannot dominate the
result. Average linkage (UPGMA) is a standard, uncontroversial pairing
with correlation distance.

For a clone whose genome already matches the additive expectation almost
everywhere, correlating tightly with both parents is close to guaranteed
by construction and not, on its own, a strong result. Clustering is more
informative for a clone with substantial non-additive deviation: whether
it still clusters with its own two parents despite that deviation is a
real, non-trivial check of whether overall genomic architecture is
preserved despite widespread local non-additivity.


In [ ]:
def plot_hierarchical_clustering(cn_df, fusion_pairs, allowed_chroms,
                                 metric="correlation", method="average", cbar_label="Copy number",
                                 vmin=0.0, vmax=12.0, name=None, out_dir=None):
    """
    Hierarchical clustering of all 8 samples (parental controls and fusion
    clones) on their genome-wide copy number profiles, to check whether
    each fusion clone clusters with its own two parents.

    Correlation distance (not Euclidean) is used because parental and
    fusion samples sit at very different baseline ploidies (see the
    per-sample complexity diagnostic in Section 3.4): correlation distance
    clusters on the pattern of relative gains and losses along the genome
    rather than on absolute copy number level, so ploidy differences alone
    cannot dominate the clustering. Average linkage (UPGMA) is a standard,
    uncontroversial pairing with correlation distance. Only rows (samples)
    are clustered; columns (genomic bins) stay in genomic order so the
    chromosome axis remains interpretable.

    Each sample is colored by which of the three FUSION_PAIRS trios it
    belongs to (a fusion clone and its own two parents share a color). The
    copy number heatmap uses a diverging blue-white-red colormap over
    [vmin, vmax] -- white falls at the midpoint of that range, blue below
    it, red above. Set vmin/vmax directly for your data (e.g. vmin=0,
    vmax=12 puts white at CN=6). Values outside [vmin, vmax] saturate to
    the nearest end color.

    Uses seaborn's clustermap rather than a hand-built matplotlib
    dendrogram + heatmap, since correctly aligning a dendrogram, a row
    color strip, and a heatmap by leaf order is easy to get subtly wrong
    by hand -- this is exactly what clustermap is designed to do reliably.
    """
    samples, trio_of_sample = [], {}
    for fusion, (parent_a, parent_b) in fusion_pairs.items():
        for s in (parent_a, parent_b, fusion):
            if s not in trio_of_sample:
                samples.append(s)
                trio_of_sample[s] = fusion

    chrom_order = {chrom: i for i, chrom in enumerate(allowed_chroms)}
    plot_df = cn_df.copy()
    plot_df["_chrom_sort"] = plot_df["Chromosome"].map(chrom_order)
    plot_df = plot_df.sort_values(["_chrom_sort", "Start"]).reset_index(drop=True)

    data = plot_df[samples].T.astype(float)
    n_before = data.shape[1]
    data = data.loc[:, np.isfinite(data.to_numpy()).all(axis=0)]
    if data.shape[1] < n_before:
        print(f"Dropped {n_before - data.shape[1]} of {n_before} bins containing non-finite values before clustering.")

    trio_names = list(fusion_pairs.keys())
    trio_cmap = plt.get_cmap("tab10")
    trio_colors = {trio: trio_cmap(i) for i, trio in enumerate(trio_names)}
    row_colors = pd.Series([trio_colors[trio_of_sample[s]] for s in samples], index=samples)

    cmap = "RdBu_r"

    g = sns.clustermap(
        data,
        cmap=cmap,
        vmin=vmin, vmax=vmax,
        metric=metric, method=method,
        row_cluster=True, col_cluster=False,
        row_colors=row_colors,
        figsize=(16, max(6, len(samples) * 0.6)),
        cbar_pos=None,
        dendrogram_ratio=(0.14, 0),
        colors_ratio=0.03,
    )
    g.ax_col_dendrogram.set_visible(False)
    g.ax_row_colors.set_xticks([])
    g.ax_row_colors.tick_params(bottom=False, labelbottom=False)
    # Shrink the main clustermap block to the left x% of the figure so the
    # colorbar and legend have their own dedicated space on the right,
    # rather than being manually placed on top of the heatmap's own extent.
    g.fig.subplots_adjust(right=0.8)

    chrom_list = plot_df["Chromosome"].values
    tick_positions, tick_labels = [], []
    for chrom in allowed_chroms:
        idxs = np.where(chrom_list == chrom)[0]
        if len(idxs) == 0:
            continue
        tick_positions.append(idxs.mean())
        tick_labels.append(chrom)
    g.ax_heatmap.set_xticks(tick_positions)
    g.ax_heatmap.set_xticklabels(tick_labels, rotation=90)
    g.ax_heatmap.set_xlabel("Chromosome")
    current_ytick_labels = [t.get_text() for t in g.ax_heatmap.get_yticklabels()]
    g.ax_heatmap.set_yticklabels([get_label(s) for s in current_ytick_labels])

    cbar_ax = g.fig.add_axes([0.9, 0.2, 0.02, 0.35])
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cbar = g.fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax, label=cbar_label)
    cbar.set_ticks([vmin, (vmin + vmax) / 2, vmax])
    cbar.ax.tick_params(labelsize=FONT_SIZE - 2)

    legend_patches = [mpatches.Patch(color=trio_colors[t], label=get_label(t)) for t in trio_names]
    g.fig.legend(handles=legend_patches, title="Fusion trio", loc="upper right",
                bbox_to_anchor=(0.99, 0.98), bbox_transform=g.fig.transFigure,
                frameon=True, framealpha=0.95)

    g.fig.suptitle(f"Hierarchical clustering of copy-number profiles ({metric} / {method})", y=1.02)

    if name:
        save_fig(g.fig, name, out_dir=out_dir)
    plt.show()

    ordered_samples = [samples[i] for i in g.dendrogram_row.reordered_ind]
    return ordered_samples, g

ordered_samples, clustergrid = plot_hierarchical_clustering(
    cn_df, FUSION_PAIRS, ALLOWED_CHROMS,
    metric="correlation", method="average",vmin=0, vmax=8,
    name="hierarchical_clustering_CN_correlation", out_dir=cross_clone_dir
)
print("Leaf order (top to bottom in the dendrogram):", ordered_samples)

In [ ]:
additive_sample_names = sort_sample_cols(sorted(set(PARENTAL_SAMPLES) | set(FUSION_PAIRS.keys())))
log2_cnr_df = merge_freec_ratio_data(FREEC_BASE_DIR, ALLOWED_CHROMS,
                                     additive_sample_names, cn_df)
log2_cnr_path = os.path.join(BASE_OUTPUT_DIR, "additive_log2_ratio_matrix.tsv")
log2_cnr_df.to_csv(log2_cnr_path, sep="\t", index=False)
print(f"Saved: {log2_cnr_path}  ({len(log2_cnr_df)} bins x {len(additive_sample_names)} samples)")

ordered_samples, clustergrid = plot_hierarchical_clustering(
    log2_cnr_df, FUSION_PAIRS, ALLOWED_CHROMS, cbar_label=r"log$_2$ CNR",
    metric="correlation", method="average",vmin=-1.0, vmax=1.0,
    name="hierarchical_clustering_log2_correlation", out_dir=cross_clone_dir
)

### 2.3 Per-Chromosome Heatmap - Log2 CNR

In [ ]:
for chrom in ALLOWED_CHROMS:
    plot_chromosome_heatmap_log2(
        log2_cnr_df, chromosome=chrom, sample_cols=additive_sample_names,
        value_label=r"Log$_2$ CNR", vmin=-1, vmax=1,
        output_prefix="chr_log2cnr",
    )